# 07 — ACDC / MRzero for spiral imaging with 32-echo signal modelling

This notebook adapts version 06 to the measured **spiral GRE acquisition** in
`acquisition_032_(07)`. The actual Pulseq ADC trajectory is used throughout:
no Cartesian lines are invented and no ADC sample is rounded to a grid.
The 32 ACDC channels receive one command vector **per spiral revolution**.
Turn boundaries are the `ext1` trigger times encoded in the sequence. The
current-response model remains continuous across those boundaries.

The reference GRE and the 32-echo Cartesian MEGRE mapping acquisition retain
their own sequence, native matrix, physical coordinates, slice position and
orientation. The measured GRE is the image optimization target; the zero-B0
simulation is a separate diagnostic. The following version-06 features are retained:

- Independent receiver-noise calibration, SNR-based support and early-echo B0.
- Independent complex receiver encoding from the MEGRE fit, with one complex
  reconstruction per receiver followed by RSS image combination.
- Complex 32-echo model selection on held-out echoes, finite-voxel integration,
  unresolved coherence, and a validated GRE amplitude/contrast prior.
- Physical rotation/mirrors, offset and oblique slice geometry, 3D coil-map
  sampling, thickness subspins and field constraints on voxel corners.
- Static initialization, exact amplifier response, disabled physical channels,
  current/field/hardware limits, DAC checks and physical residual-field guidance.
- Bounded baseline calibration, two-sided signal-disagreement diagnostics,
  live optimization and saved image history.
- Image/static/dynamic waveform exports, ADC-measurement schedules, actual CSV
  preview and measured corrected-scan comparison with one fixed baseline gain.

Spiral reconstruction uses the same fixed non-Cartesian operator for measured
and simulated samples and its exact Hermitian adjoint for optimization. Its
sampling-density weights and normalization are frozen before optimization.
Spiral off-resonance generally produces blurring and signal cancellation rather
than the simple phase-encoding displacement of EPI. The phase evolution is
simulated at the actual ADC times, including each coil's response history.

A single 2D MEGRE map does not measure the B0 gradient through the slice.
Unresolved 32-echo attenuation remains an effective, current-independent factor;
this notebook does not assume it can be recovered by ACDC.

Run all cells in order after restarting the kernel when modules or acquisition
settings change. Every output from this version is stored under a new
`results_experiment_spiral07/mrzero07_*` session; existing version-06 results are preserved.


In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT=Path.cwd() if (Path.cwd()/"Experiment").is_dir() else Path.cwd().parent
for folder in (PROJECT_ROOT/"Experiment",PROJECT_ROOT/"MRzero_validation"):
    if str(folder) not in sys.path:sys.path.insert(0,str(folder))
from twin_inputs import prepare_echo_object
from spiral_inputs import read_spiral_inputs
from echo_shim_inputs import prepare_echo_shim_inputs
from experiment_echo import EchoSubspinACDCModel
from experiment_voxel import FiniteVoxelEchoACDCModel
from experiment_multiecho import MultiEchoVoxelACDCModel
from experiment_spiral import SpiralReceiverACDCModel
from spiral_receive import prepare_spiral_receive_model,fixed_spiral_object_checks
from spiral_encoding import apply_spiral_encoding,reconstruction_acquisition,calibration_acquisition
from spiral_trajectory_fit import fit_spiral_gradient_delays
from multiecho_object import prepare_complex_echo_model,multiecho_comparison
from reference_echo_prior import apply_reference_amplitude_prior
from measured_image_target import configure_measured_gre_target,relative_image_mse
from b0_regularization import regularize_b0_local_plane
from inplane_geometry import inplane_matrix,FrameReconstruction
# Shared legacy module name; the point twin accepts the actual spiral ADC coordinates.
from epi_twin import NativeEPITwin as NativePointTrajectoryTwin
from spiral_notebook_tools import plot_spiral_trajectory, set_turn_axis_labels
from twin_geometry import compare_magnitude
from acdc_gpu_field import solve_field_gpu
from acdc_gpu import select_device
from experiment_optimization import optimize_mrzero_image
from field_guidance import FieldGuidance
from live_optimization import LiveOptimizationPlot
from optimization_image_history import OptimizationImageHistory
from acdc_waveform_plot import plot_acdc_waveform
from spiral_measured_comparison import compare_measured_scan_set
from export_acdc_waveform import quantize_currents,export_waveform,export_zero_time_copy,validate_waveform_settings
from shim_export_timing import sequence_trigger_timing,waveform_edges_for_origin
from field_shim_export import export_field_shim_waveforms
from read_acdc_b0 import read_b0_sequence
from read_spiral_sequence import read_spiral_sequence

# Independent measured inputs: the imaging trajectory is spiral; GRE/MEGRE stay Cartesian.
INPUT_DIR=PROJECT_ROOT/"acquisition_032_(07)"
SPIRAL_SEQUENCE_NAME="gre_spiral_128x128_fov200mm_nav3_cimax_turn_triggers_v1.seq"
SPIRAL_SCAN_PATH="meas_MID00193_FID50216_spiral_scan.dat"
REFERENCE_SEQUENCE_NAME="gre_128x128_fov200mm_flip15deg_tr15ms.seq"
REFERENCE_SCAN_PATH="meas_MID00196_FID50217_reference.dat"
B0_SEQUENCE_NAME="megre_b0_s0_t2star_128x128_fov200mm_32echo.seq"
B0_SCAN_PATH="meas_MID00197_FID50218_b0_map.dat"

CALIBRATION_PATH=PROJECT_ROOT/"data/ACDC_3T.npz"
WAVEFORM_OUTPUT_DIR=INPUT_DIR/"waveform"
OUTPUT_DIR=PROJECT_ROOT/"MRzero_validation/results_experiment_spiral07"/datetime.now().strftime("mrzero07_%Y%m%d_%H%M%S_%f")
print("Shared coil calibration:",CALIBRATION_PATH.resolve())
print("Waveform directory:",WAVEFORM_OUTPUT_DIR.resolve())


## 1. Acquisition, geometry, signal model and hardware settings

Set the three measured inputs independently. `SPIRAL_SCAN_PATH`,
`REFERENCE_SCAN_PATH` and `B0_SCAN_PATH` accept a filename relative to `INPUT_DIR`
or an absolute path. Their sequences live in `INPUT_DIR/sequences` unless an
absolute `.seq` path is supplied. The scan must match its actual sequence;
changing a sequence filename cannot convert another trajectory or matrix.

**Broken channels.** `DISABLED_CHANNELS=[9]` retains the current version-06
configuration: physical output 9 is held at exactly zero during initialization,
image/field optimization, DAC checks and exports. Use `[]` to enable every output.
These are physical output numbers in `WAVEFORM_PARAMETERS['channel_map']`,
not Python indices. Physical output 9 is CSV column `ch8_A` with the default map.
The other channels are fitted again after a channel is disabled.

**Physical alignment and display frame.** `Q=R(angle) @ diag(mirror signs)`
maps image-plane coordinates to the calibrated coils. Mirrors precede rotation.
`P` changes the common representation after physical alignment; it changes both
spins and k coordinates, so it does not move the object relative to the coils.
Two mirrors are equivalent to a 180-degree rotation. The measured Twix slice
centre and orientation place the slice and thickness subspins in patient space.

**32-echo object.** The early `B0_FIELD_ECHO_COUNT` echoes initialize B0.
`ECHO_SIGNAL_MODEL='auto'` uses the complex receiver data for real acquisitions.
The last `COMPLEX_VALIDATION_ECHOES` echoes choose among exponential, Gaussian
and signed-sinc coherence models, before refitting the selected family to all
32 echoes. Weak and zero echoes remain observations. `OBJECT_AMPLITUDE_SOURCE='joint'`
tests the measured GRE structural prior with a smooth inter-acquisition contrast
field; it is retained only when held-out echo prediction improves. The GRE
measurement remains unchanged and serves independently as the image target.
`'echo32'`, `'auto'` and `'reference'` retain the alternative amplitude choices.
A single GRE does not separately identify quantitative PD, T1 or B1.

**Noise and support.** Inspect the fixed background corners or supply a native
boolean `NOISE_MASK`. Noise calibration precedes object masking; contaminated
background is not silently replaced with the object complement. SNR and spatial
component checks define the supported object. The default requires a usable
noise estimate. The complex model reports quality limits without replacing
signal losses with zero decay. `RATE_MODE` applies to the legacy RSS comparison.

**Simulation.** `SIMULATION_MODEL='auto'` selects finite voxels for measured data
and point spins for point-generated synthetic inputs. Finite voxels integrate
known in-plane B0 and ACDC gradients exactly once. `SUBSPINS_Z` samples the actual
3D coil field across the slice; the measured B0 is constant through its thickness.
The model excludes incomplete parents without silently renormalizing their signal.

**Receive model.** `SPIRAL_RECEIVE_MODEL='auto'` transfers the independently
measured complex MEGRE receiver amplitudes and initial phase to the spiral
model. Each receiver signal is reconstructed separately; RSS combination
occurs only after reconstruction. Per-pixel unit-RSS normalization preserves
the existing object amplitude. `'complex_echo'` requires those data;
`'rss_proxy'` requests the historical one-receiver RSS approximation.
Twix receiver IDs establish channel order when available. The 32 receive
channels measure signal and are distinct from the 32 ACDC current outputs;
disabling an ACDC output does not remove a receive channel. `RECEIVER_CHUNK`
bounds receiver work batches without adding native MRzero spin histories.
The transfer assumes stable spatial receive/initial phase between MEGRE and
spiral measurements and cannot establish an exact scanner model.

**Computation.** `GPU_REQUIRED=True` requires actual CUDA execution. For an
intentional CPU run set it to `False` and use `GPU_PROFILE='cpu'`. Device selection,
precision, native-cache storage and sample chunks are reported separately.
Restart and rebuild the model after changing the device or modules. See
`../docs/SERVER_SETUP.md` for the environment setup.
`SPIRAL_RECONSTRUCTION_METHOD='nufft'` uses the fast nonuniform Fourier operator;
`'direct'` is the bounded exact Fourier reference for small validation cases.
`SPIRAL_DENSITY_COMPENSATION='pipe_menon'` estimates sampling-density weights
once and reuses them for every measured/simulated reconstruction. Neither the
trajectory nor those weights are fitted during current optimization.
Version-06 CUDA graph
capture remains disabled so continuous hardware projection is retained.

**Image target and field guidance.** The measured GRE is interpolated onto the
spiral reconstruction grid in physical coordinates without extrapolation. A
fixed simulation-to-baseline-to-GRE gain and a fixed mask are fitted once.
`OPTIMIZATION_REFERENCE_THRESHOLD=0` includes the entire common coverage,
including dark pixels; a positive value requests a fixed GRE-threshold ROI.
`FIELD_GUIDANCE_WEIGHT` adds a physical residual-B0 penalty with normalization
`FIELD_GUIDANCE_SCALE_HZ`. It does not raise the available hardware field limit.

**Synchronization and exports.** The spiral sequence's verified turn triggers
are the command boundaries. The same origin, physical channel mapping and
polarity serve image, static/dynamic and ADC exports. Output files are numbered
rather than overwritten. No notebook cell programs the hardware controller.

The spiral forward model can estimate two bounded native gradient delays from
training pixels. Validation selects the candidate and a reserved test subset
gates application after full nonlinear signal recomputation. This changes
only simulated gradient encoding; both images retain the nominal reconstruction.


In [ ]:
# Physical image-to-coil alignment; mirrors are applied before rotation.
ALIGNMENT_ANGLE_DEG=90
ALIGNMENT_MIRROR_X=False
ALIGNMENT_MIRROR_Y=False


# Joint coordinate representation after alignment; no relative object/coil motion.
FRAME_ANGLE_DEG=90.0
FRAME_MIRROR_X=False
FRAME_MIRROR_Y=False


# Native MEGRE support, independent background noise and complex temporal model.
OBJECT_THRESHOLD_FRACTION=0.04
BACKGROUND_CORNER_M=0.085
NOISE_MASK=None
OBJECT_MIN_SNR=5.0
OBJECT_MIN_COMPONENT_VOXELS=4
REQUIRE_NOISE_CALIBRATION=True
B0_FIELD_ECHO_COUNT=6
ECHO_SIGNAL_MODEL="auto"
COMPLEX_FIT_BATCH_SIZE=256
COMPLEX_FIT_ITERATIONS=60
COMPLEX_VALIDATION_ECHOES=6
COMPLEX_MAX_WIDTH_HZ=120.
OBJECT_AMPLITUDE_SOURCE="joint"
OBJECT_CONTRAST_SIGMA_MM=10.0
REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT=0.02
SIMULATION_MODEL="auto"
B0_WEIGHTING="auto"
BASELINE_CALIBRATION="auto"
# Spiral calibration excludes the EPI odd/even phase parameter.
CALIBRATION_BOUNDS={"frequency_offset_Hz":30.,"readout_shift_mm":2.,"phase_shift_mm":2.}
COMPARE_POINT_BASELINE=True
B0_MODE="regularized"
B0_SIGMA_PIXELS=1.0
RATE_MODE="guarded"
SUBSPINS_Z=8

# Actual execution device, cache budget and chunk sizes.
GPU_PROFILE="auto"
GPU_REQUIRED=True
GPU_PRECISION="auto"
GPU_MEMORY_FRACTION=0.60
SAMPLE_CHUNK=None
RECONSTRUCTION_CHUNK=256
# Fixed non-Cartesian reconstruction and sampling-density model.
SPIRAL_RECONSTRUCTION_METHOD="nufft"
SPIRAL_DENSITY_COMPENSATION="pipe_menon"
# Independent complex receive encoding; these receivers are not ACDC outputs.
SPIRAL_RECEIVE_MODEL="auto"
RECEIVER_CHUNK=4
# Fit two bounded gradient delays; measured reconstruction and ACDC timing stay fixed.
SPIRAL_GRADIENT_DELAY_CALIBRATION=True
SPIRAL_MAX_GRADIENT_DELAY_US=20.0

# Hardware limits apply to actual currents and finite-voxel corners.
MAX_CURRENT_A=3.0
# Physical output numbers (1..32), not Python indices; output 9 remains disabled from version 06.
DISABLED_CHANNELS=[9]
TOTAL_CURRENT_LIMIT_A=30.0
FIELD_LIMIT_HZ=1000.0
CURRENT_RESPONSE={"enabled":True,"tau_s":133e-6/np.log(100.)}
HARDWARE={"min_update_interval_s":8e-6,"dac_resolution_A":0.0006,
          "max_slew_A_per_s":None,"avg_abs_current_A":None,
          "coil_R_ohm":None,"coil_L_H":None,"max_voltage_V":None,"max_coil_power_W":None}

# Static initialization and a separate physical residual-field guide.
IMAGE_INITIALIZATION="static"

# Weight multiplies the dimensionless residual-field penalty; zero is image-only.
FIELD_GUIDANCE_WEIGHT=0.05
# Fixed normalization in Hz; this is not the available correction-field limit.
FIELD_GUIDANCE_SCALE_HZ=100.0
MAX_ITERATIONS=1000
LEARNING_RATE_A=0.01
PATIENCE=100
TARGET_MSE=None
OPTIMIZATION_REFERENCE_THRESHOLD=0.0

LIVE_UPDATE_EVERY=20
RUN_OPTIMIZATION=True


# Local exports; this notebook does not program the hardware controller.
EXPORT_WAVEFORM=True
EXPORT_FIELD_SHIM_WAVEFORMS=True
FIELD_SHIM_RASTER_S=10e-6
FIELD_SHIM_MAX_ITERATIONS=80
CSV_TIME_ORIGIN="first_imaging_trigger"
TRIGGER_CHANNEL="ext1"


EXPORT_ADC_WAVEFORM=True
WAVEFORM_ADC_INTERVAL_S=0.000050
WAVEFORM_ADC_DURATION_S=None
WAVEFORM_PARAMETERS={"channel_map":list(range(1,33)),"channel_polarity":[-1]*32,
    "max_current_A":MAX_CURRENT_A,"total_current_limit_A":TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s":8e-6,"disabled_channels":DISABLED_CHANNELS}

WAVEFORM_PARAMETERS=validate_waveform_settings(WAVEFORM_PARAMETERS,len(WAVEFORM_PARAMETERS["channel_map"]))
CHANNEL_MAX_CURRENT_A=np.asarray(WAVEFORM_PARAMETERS["max_current_A"],dtype=float)

CALIBRATION_PARAMETERS={"units":"Hz/A","registration_confirmed":True,
    "patient_to_source_m":np.eye(4).tolist(),
    "source_sha256":"bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note":"Previously confirmed Bay5 placement; image orientation is configured above."}


# None compares every *scan.dat in INPUT_DIR/results with one fixed baseline gain.
CORRECTED_SCAN_NAMES=None
NORMALIZE_COMPARISON=True
COMPARISON_REFERENCE_THRESHOLD=0.1
WAVEFORM_PREVIEW_CHANNEL=None


In [ ]:
def table(rows,headings=("Parameter","Value")):
    display(Markdown("| "+" | ".join(headings)+" |\n| "+" | ".join(["---"]*len(headings))+" |\n"+
        "\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in rows)))

def show(fig):
    plt.show();plt.close(fig)

def image_physical(ax,image,X,Y,title,cmap="gray",**options):
    artist=ax.pcolormesh(X*1000,Y*1000,np.asarray(image),shading="auto",cmap=cmap,**options)
    ax.set(title=title,xlabel="X [mm]",ylabel="Y [mm]",aspect="equal")
    return artist

def json_safe(value):
    if isinstance(value,dict):return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [json_safe(v) for v in value]
    if isinstance(value,np.ndarray):return json_safe(value.tolist())
    if isinstance(value,np.generic):return json_safe(value.item())
    if isinstance(value,Path):return str(value)
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

angles={"alignment_angle_deg":ALIGNMENT_ANGLE_DEG,"alignment_mirror_x":ALIGNMENT_MIRROR_X,
    "alignment_mirror_y":ALIGNMENT_MIRROR_Y,"frame_angle_deg":FRAME_ANGLE_DEG,
    "frame_mirror_x":FRAME_MIRROR_X,"frame_mirror_y":FRAME_MIRROR_Y}
Q=inplane_matrix(ALIGNMENT_ANGLE_DEG,ALIGNMENT_MIRROR_X,ALIGNMENT_MIRROR_Y)
P=inplane_matrix(FRAME_ANGLE_DEG,FRAME_MIRROR_X,FRAME_MIRROR_Y)
O=P@Q
table(list(angles.items()))
table([("Disabled physical outputs",WAVEFORM_PARAMETERS["disabled_channels"]),
       ("Active channel count",int(np.count_nonzero(CHANNEL_MAX_CURRENT_A)))])
print("Image-to-coil matrix Q:\n",Q,"\nComputational-frame matrix O=P@Q:\n",O)
print("Session output:",OUTPUT_DIR)


print("Kernel Python:",sys.executable)
print("PyTorch / runtime CUDA :",torch.__version__,"/",torch.version.cuda)
execution_device,execution_dtype,execution_report=select_device(
    GPU_PROFILE,GPU_PRECISION,require_cuda=GPU_REQUIRED)
table([("Compute device",str(execution_device)),
       ("GPU",execution_report["gpu_name"]),
       ("Precision",execution_report["precision"]),
       ("CUDA required",GPU_REQUIRED)])


## 2. Verify the spiral trajectory and prepare the 32 echoes

The sequence is parsed before reading Twix. Inspect the actual k-space path,
ADC time range, turn triggers, number of commands, and reconstruction grid.
A trigger marks the next ACDC state even when it occurs between ADC centres;
no sample is duplicated or discarded at a turn boundary. Navigator readouts
are not treated as spiral turns.

GRE and MEGRE remain Cartesian and are read using their validated acquisition
adapters. Their native FOV, slice orientation and centre are checked against the
spiral scan. The independent GRE reference is displayed before any 32-echo fit.
The early echoes initialize B0; all 32 echoes contribute to the temporal model.

For the supplied acquisition, the MEGRE echoes end at about 58.5 ms while the
spiral ADC continues to about 153.9 ms after RF. The temporal prediction therefore
extrapolates for the late outer-k samples; inspect the coverage table and echo
validation plots before interpreting a predicted signal recovery.


In [ ]:
import warnings
import importlib
import read_acdc_b0 as b0_reader

b0_reader=importlib.reload(b0_reader)
read_b0_sequence=b0_reader.read_b0_sequence

def selected_sequence_path(name):
    if name is None:
        return None
    path=Path(name)
    return path if path.is_absolute() else INPUT_DIR/"sequences"/path

selected_spiral_sequence=selected_sequence_path(SPIRAL_SEQUENCE_NAME)
selected_b0_sequence=selected_sequence_path(B0_SEQUENCE_NAME)
selected_reference_sequence=selected_sequence_path(REFERENCE_SEQUENCE_NAME)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    b0_sequence_report=read_b0_sequence(selected_b0_sequence)
    spiral_preview,spiral_timing_preview=read_spiral_sequence(selected_spiral_sequence)
if b0_sequence_report["echo_count"] != 32 or tuple(b0_sequence_report["image_shape"]) != (128,128):
    raise ValueError("Notebook 07 requires 32 mapping echoes and a 128×128 matrix.")
if not 2 <= B0_FIELD_ECHO_COUNT <= b0_sequence_report["echo_count"]:
    raise ValueError("B0_FIELD_ECHO_COUNT must select between 2 and 32 echoes.")
expected_echo_times=np.asarray(b0_sequence_report["TE_s"])
spiral_relative_times=np.asarray(spiral_preview["time_s"])-np.asarray(spiral_preview["rf_time_s"])
sequence_assessment={"B0_sequence":B0_SEQUENCE_NAME,"B0_expected_echo_count":32,
    "B0_field_echo_count":B0_FIELD_ECHO_COUNT,"echo_times_s":expected_echo_times,
    "target_TE_s":float(spiral_preview["TE_s"]),
    "spiral_ADC_range_s":[float(spiral_relative_times.min()),float(spiral_relative_times.max())],
    "extrapolation_to_TE_s":max(0.,float(spiral_preview["TE_s"]-expected_echo_times[-1])),
    "extrapolation_to_last_ADC_s":max(0.,float(spiral_relative_times.max()-expected_echo_times[-1]))}
table([("Mapping sequence",B0_SEQUENCE_NAME),
       ("Echoes / ADC matrix",f"{b0_sequence_report['echo_count']} / {b0_sequence_report['image_shape']}"),
       ("First / last TE [ms]",expected_echo_times[[0,-1]]*1000),
       ("Mapping TR [ms]",b0_sequence_report["TR_s"]*1000),
       ("Mapping duration [s]",b0_sequence_report["sequence_duration_s"]),
       ("Spiral TE [ms]",spiral_preview["TE_s"]*1000),
       ("First / last ADC after RF [ms]",np.array(sequence_assessment["spiral_ADC_range_s"])*1000),
       ("Last echo to spiral TE extrapolation [ms]",sequence_assessment["extrapolation_to_TE_s"]*1000),
       ("Last echo to final ADC extrapolation [ms]",sequence_assessment["extrapolation_to_last_ADC_s"]*1000)])
print("Mapping Twix:",B0_SCAN_PATH if B0_SCAN_PATH is not None else "one unique *b0_map.dat; 32 echoes required")
print("Reference Twix:",REFERENCE_SCAN_PATH if REFERENCE_SCAN_PATH is not None else "unique *reference*.dat")
print("Reference sequence:",selected_reference_sequence or "compatible auto selection, otherwise nominal Cartesian Twix")
print("Spiral Twix:",SPIRAL_SCAN_PATH if SPIRAL_SCAN_PATH is not None else "one *scan.dat excluding reference/B0")

fig,spiral_trajectory_report=plot_spiral_trajectory(spiral_preview,selected_spiral_sequence)
show(fig)
sequence_assessment["spiral_trajectory"]=spiral_trajectory_report


In [ ]:
import importlib
import twin_inputs
import spiral_inputs
twin_inputs=importlib.reload(twin_inputs)
spiral_inputs=importlib.reload(spiral_inputs)
read_spiral_inputs=spiral_inputs.read_spiral_inputs
prepare_echo_object=twin_inputs.prepare_echo_object

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r"^File version 1\.5\.1 is higher than installed package version 1\.5\.0\. This may cause parsing errors\.$",
                            category=UserWarning, module=r"^pypulseq\.Sequence\.read_seq$")
    inputs=read_spiral_inputs(INPUT_DIR,
        scan_sequence_path=selected_spiral_sequence,scan_path=SPIRAL_SCAN_PATH,
        reference_sequence_path=selected_reference_sequence,reference_scan_path=REFERENCE_SCAN_PATH,
        b0_sequence_path=selected_b0_sequence,
        b0_scan_path=B0_SCAN_PATH,b0_echo_count=B0_FIELD_ECHO_COUNT,
        reconstruction_device=execution_device,reconstruction_dtype=execution_dtype,
        reconstruction_method=SPIRAL_RECONSTRUCTION_METHOD,density_compensation=SPIRAL_DENSITY_COMPENSATION)

print("GRE reference:",inputs["paths"]["reference"].resolve())
print("GRE sequence actually used:",inputs["paths"].get("reference_sequence","none — nominal Cartesian Twix trajectory"))
table([("Native reference matrix",np.asarray(inputs["gre"]["magnitude"]).shape),
       ("Reference TE [ms]",float(inputs["gre"].get("TE_s",np.nan))*1000),
       ("Reference TR [ms]",float(inputs["gre"].get("TR_s",np.nan))*1000),
       ("Reference validation",inputs.get("reference_sequence_validation",{})),
       ("Slice geometry checks",inputs.get("geometry_checks",[]))])
reference_gre=inputs["gre"]
reference_magnitude=np.asarray(reference_gre["magnitude"])
reference_rows,reference_columns=reference_magnitude.shape
fig,ax=plt.subplots(figsize=(8,6),layout="constrained")
reference_artist=image_physical(ax,reference_magnitude,
    reference_gre["X_m"],reference_gre["Y_m"],
    f"Measured GRE reference — {reference_rows}×{reference_columns}\n"
    f"{inputs['paths']['reference'].name}",vmin=0)
fig.colorbar(reference_artist,ax=ax,label="RSS magnitude [a.u.]")
show(fig)

if len(inputs["echo_times_s"]) != 32:
    raise ValueError("The selected Twix file does not contain the expected 32 echoes.")
table(list(inputs["scan_diagnostics"]["continuous_reconstruction"].items()),("Spiral reconstruction","Value"))
fit=prepare_echo_object(inputs,OBJECT_THRESHOLD_FRACTION,BACKGROUND_CORNER_M,
    allow_uncalibrated_noise=not REQUIRE_NOISE_CALIBRATION,noise_mask=NOISE_MASK,
    support_min_snr=OBJECT_MIN_SNR,min_component_voxels=OBJECT_MIN_COMPONENT_VOXELS)

legacy_echo_fit=fit
if ECHO_SIGNAL_MODEL not in {"auto","complex","legacy"}:
    raise ValueError("ECHO_SIGNAL_MODEL must be auto, complex or legacy.")
input_is_synthetic=(inputs.get("manifest") or {}).get("format") in {
    "mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
complex_echo_enabled=(ECHO_SIGNAL_MODEL=="complex" or
                      (ECHO_SIGNAL_MODEL=="auto" and not input_is_synthetic))
effective_rate_mode="complex" if complex_echo_enabled else RATE_MODE
complex_echo_diagnostics=None
if complex_echo_enabled:
    echo_weighting=("echo_power" if B0_WEIGHTING=="auto" else B0_WEIGHTING)
    fit=prepare_complex_echo_model(inputs,legacy_echo_fit,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        b0_sigma_pixels=B0_SIGMA_PIXELS,b0_weighting=echo_weighting,
        batch_size=COMPLEX_FIT_BATCH_SIZE,max_iterations=COMPLEX_FIT_ITERATIONS,
        max_width_Hz=COMPLEX_MAX_WIDTH_HZ,validation_echo_count=COMPLEX_VALIDATION_ECHOES)
    fit=apply_reference_amplitude_prior(inputs,fit,
        source=OBJECT_AMPLITUDE_SOURCE,contrast_sigma_mm=OBJECT_CONTRAST_SIGMA_MM,
        device=str(execution_device),require_cuda=GPU_REQUIRED,
        validation_echo_count=COMPLEX_VALIDATION_ECHOES,
        min_relative_validation_improvement=REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        fit_options={"batch_size":COMPLEX_FIT_BATCH_SIZE,"max_iterations":COMPLEX_FIT_ITERATIONS,
                     "max_width_Hz":COMPLEX_MAX_WIDTH_HZ})
    table(list(fit["report"]["reference_amplitude_prior"].items()),("Reference amplitude contribution","Value"))
    complex_echo_diagnostics=multiecho_comparison(inputs,fit)
    table(list(fit["report"]["multi_echo"].items()),("Complex 32-echo fit","Value"))
    table([(row["region"],row["pixels"],row["legacy_RSS_RMSE_over_first_echo"],
            row["complex_RSS_RMSE_over_first_echo"]) for row in complex_echo_diagnostics["rows"]],
          ("MEGRE region","Pixels","Legacy error / first echo","Complex error / first echo"))
print("Temporal model used by MRzero:",effective_rate_mode)
noise_info=fit["report"]["noise"]
support_info=fit["report"]["support"]
raw=inputs["b0_raw"]
acquisition_native=inputs["acquisition"]
pooled=fit["pooled"]
print("Mapping Twix:",inputs["paths"]["b0"].resolve())
print("Mapping sequence:",inputs["paths"]["b0_sequence"].resolve())
print("Spiral scan without ACDC:",inputs["paths"]["scan"].resolve())
print(f"Noise: {noise_info['available_sample_count']} finite complex vectors / "
      f"{noise_info['minimum_sample_count']} required; calibrated = {noise_info['calibrated']}.")
if not noise_info["calibrated"]:
    print("Fallback: first echo, no additional R2* decay. Reason:",noise_info["fallback_reason"])
table([("Echo count read for relaxation",len(inputs["echo_times_s"])),
       ("Early echo count for B0",inputs["field_echo_count"]),
       ("Mapping TE [ms]",inputs["echo_times_s"]*1000),
       ("Spiral TE [ms]",acquisition_native["TE_s"]*1000),
       ("Independent noise pixels",int(fit["noise_mask"].sum())),
       ("Object-support pixels",int(fit["object_support"].sum())),
       ("Simulatable pixels with B0",int(fit["simulation_support"].sum())),
       ("Diagnostic pooled R2* [s⁻¹]",pooled.get("R2star_sInv")),
       ("Pooled-rate uncertainty [s⁻¹]",pooled.get("R2star_std_sInv")),
       ("Pooled rate accepted at spiral TE",pooled.get("target_reliable_mask",False)),
       ("Legacy RSS fits accepted at spiral TE",int(fit["target_reliable_mask"].sum()))])
table(list(support_info.items()),("Support diagnostics","Value"))

raw_first_rss=np.sqrt(np.sum(np.abs(raw["coil_images"][0])**2,axis=0))
accepted_rates=(np.where(fit["multi_echo"]["support_mask"],fit["multi_echo"]["residual_R2_sInv"],np.nan)
                if complex_echo_enabled else np.where(fit["target_reliable_mask"],fit["R2star_sInv"],np.nan))
fig,axes=plt.subplots(2,3,figsize=(15,9),layout="constrained")
images=[raw_first_rss,fit["support_debiased_amplitude"],fit["object_support"],
        fit["noise_mask"],inputs["field"]["b0Hz"],accepted_rates]
amplitude_title=("RSS noise-debiased amplitude" if support_info["amplitude_debiased"]
                 else "Raw RSS: uncalibrated noise")
titles=["Raw first-echo RSS",amplitude_title,"SNR-based object support",
        "Independent noise mask","Early-echo B0 [Hz]",("Complex residual decay [s⁻¹]" if complex_echo_enabled else "Local R2* accepted at spiral TE [s⁻¹]")]
for ax,image,title in zip(axes.ravel(),images,titles):
    im=image_physical(ax,image,raw["X_m"],raw["Y_m"],title,
                      cmap="RdBu_r" if "[Hz]" in title else "gray")
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)


### Current-measurement schedule: `waveform_adc.csv`

`WAVEFORM_ADC_INTERVAL_S=0.000050` requests a 50 µs measurement step. The CSV has
one numeric time column and no header. Time resets to zero at every verified
`ext1` trigger, including each spiral-turn trigger.

The default acquisition window ends at the final imaging ADC. An explicit
`WAVEFORM_ADC_DURATION_S` sets its duration after the first trigger. The next
trigger or final window boundary is excluded from the preceding segment.
The companion JSON records absolute trigger times and segment/sample indices.
The export is independent of image optimization and field-shim exports.


In [ ]:
adc_waveform_export=None
if EXPORT_ADC_WAVEFORM:
    from adc_waveform_export import export_adc_waveform
    adc_waveform_export=export_adc_waveform(
        inputs["paths"]["scan_sequence"],inputs["timing"],WAVEFORM_OUTPUT_DIR,
        interval_s=WAVEFORM_ADC_INTERVAL_S,trigger_channel=TRIGGER_CHANNEL,
        duration_s=WAVEFORM_ADC_DURATION_S)
    print("CSV ADC :",adc_waveform_export["csv_file"])
    print("ADC step [s]:",adc_waveform_export["interval_s"],
          "| samples:",adc_waveform_export["samples"],
          "| triggers :",adc_waveform_export["trigger_count"])
else:
    print("ADC export disabled.")


In [ ]:

times=np.asarray(inputs["echo_times_s"])
mean_echoes=np.asarray(fit["pooled"].get("mean_RSS_echoes",[]))
fig,ax=plt.subplots(figsize=(10,4),layout="constrained")
if len(mean_echoes)==len(times):
    ax.plot(times*1000,mean_echoes,"o",label="Mean noise-debiased amplitude in pooled ROI")
    rate=fit["pooled"].get("R2star_sInv",np.nan)
    anchor_amplitude=fit["pooled"].get("fitted_amplitude_at_reference",np.nan)
    if np.isfinite(rate) and np.isfinite(anchor_amplitude):
        prediction_time=np.linspace(times[0],max(times[-1],sequence_assessment["spiral_ADC_range_s"][1]),300)
        prediction=anchor_amplitude*np.exp(-rate*(prediction_time-times[0]))
        ax.plot(prediction_time*1000,prediction,"--",label="Pooled fit (acceptance reported in table)")
ax.axvspan(times[0]*1000,times[-1]*1000,color="tab:green",alpha=.1,label="Measured multi-echo range")
ax.axvline(acquisition_native["TE_s"]*1000,color="tab:red",label="Spiral TE")
ax.axvline(sequence_assessment["spiral_ADC_range_s"][1]*1000,color="tab:gray",linestyle=":",label="Final spiral ADC")
ax.set(xlabel="Time after RF [ms]",ylabel="Effective amplitude",title="Pooled RSS diagnostic: temporal coverage")
ax.legend(fontsize=8);show(fig)
if not complex_echo_enabled and not fit["target_reliable_mask"].any():
    print("No local fit is accepted at spiral TE: check late-echo SNR and diagnostics before interpreting correction.")


### 2a. Check signal holes and late-echo prediction

For each selected MEGRE echo, compare the measured RSS, historical prediction,
complex prediction and signed error at a common scale. The descriptive hole
mask identifies pixels bright at the first echo and below 25% at echo 15;
it does not change support or fitting weights. Post-refit plots are distinct
from held-out prediction errors saved during model selection. Every model is
shown on the same measured B0 support.


In [ ]:
if complex_echo_enabled:
    ce=fit["multi_echo"]
    echo_indices=[0,7,14,19,25,31]
    measured=fit["debiased_RSS"]
    predicted=ce["predicted_RSS"]
    previous=complex_echo_diagnostics["legacy_predicted_RSS"]
    vmax=float(np.nanpercentile(measured[0][ce["support_mask"]],99))
    fig,axes=plt.subplots(4,len(echo_indices),figsize=(21,13),layout="constrained")
    for column,index in enumerate(echo_indices):
        for row,(image,label) in enumerate([(measured[index],"Measured"),
            (previous[index],"Legacy model"),(predicted[index],"Complex fit"),
            (predicted[index]-measured[index],"Fit error")]):
            signed=row==3
            axes[row,column].set_facecolor("black")
            image_physical(axes[row,column],np.where(ce["support_mask"],image,np.nan),raw["X_m"],raw["Y_m"],
                f"{label} : echo {index+1} / {inputs['echo_times_s'][index]*1000:.1f} ms",
                cmap="RdBu_r" if signed else "gray",vmin=-vmax if signed else 0,vmax=vmax)
    show(fig)
    regions=[("Measured support",ce["support_mask"]),
             ("Signal holes at echo 15",complex_echo_diagnostics["hole_mask"])]
    fig,axes=plt.subplots(1,2,figsize=(13,4),layout="constrained")
    for ax,(label,region) in zip(axes,regions):
        if not region.any():
            ax.set_title(label+" : no pixels");continue
        scale=np.maximum(fit["first_echo_amplitude"][region],np.finfo(float).tiny)
        for data,name,style in [(measured,"Measured","o"),(previous,"Legacy model","--"),
                                (predicted,"Complex fit","-")]:
            ax.plot(inputs["echo_times_s"]*1000,np.median(data[:,region]/scale[None],axis=1),
                    style,label=name)
        ax.axvspan(inputs["echo_times_s"][-COMPLEX_VALIDATION_ECHOES]*1000,
                   inputs["echo_times_s"][-1]*1000,color="tab:green",alpha=.1,
                   label="Held-out echoes before refit")
        ax.set(title=label,xlabel="TE [ms]",ylabel="Amplitude / first echo")
        ax.legend(fontsize=8)
    show(fig)


## 3. Place the object and sample the 3D coil calibration

Amplitude and B0 remain attached to the same native positions. `Q` maps the
image plane to the coils and `P` changes the common computational frame.
Every spiral ADC keeps its measured order, time and actual k coordinate.
The image arrays are not rotated, interpolated or resampled for alignment.

Each retained native pixel has `SUBSPINS_Z` positions through the measured slice
thickness. The calibrated 3D ACDC field is sampled at their actual patient-space
positions, including the Twix offset and obliquity. Incomplete parents are excluded
without signal renormalization. Field limits cover all retained subspins and,
for finite voxels, their in-plane corners.

Complex receiver weights are attached to the same retained native parents.
Their excitation-time phase is restored from the fitted MEGRE reference echo
so native MRzero evolves the measured B0 exactly once. The fixed receiver
weights are repeated through each parent's thickness children. They are not
fit to the spiral baseline and do not alter object amplitude or support.


In [ ]:


manifest_format=(inputs.get("manifest") or {}).get("format")
is_point_synthetic=manifest_format in {"mrzero_synthetic_acquisition","mrzero_synthetic_twix"}
if SIMULATION_MODEL not in {"auto","point","finite_voxel"}:
    raise ValueError("SIMULATION_MODEL must be auto, point or finite_voxel.")
if B0_WEIGHTING not in {"auto","uniform","echo_power"}:
    raise ValueError("B0_WEIGHTING must be auto, uniform or echo_power.")
effective_simulation_model=(("point" if is_point_synthetic else "finite_voxel")
                            if SIMULATION_MODEL=="auto" else SIMULATION_MODEL)
effective_b0_weighting=(("uniform" if is_point_synthetic else "echo_power")
                        if B0_WEIGHTING=="auto" else B0_WEIGHTING)
if complex_echo_enabled and effective_simulation_model!="finite_voxel":
    raise ValueError("The complex fit with known gradients requires SIMULATION_MODEL=auto or finite_voxel.")
print("Spiral model:",effective_simulation_model,"| B0 weighting:",effective_b0_weighting)
prepared=prepare_echo_shim_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
    **angles,subspins_z=SUBSPINS_Z,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
    b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
    b0_weighting=effective_b0_weighting,inplane_voxel=effective_simulation_model=="finite_voxel")
prepared=prepare_spiral_receive_model(prepared,fit,inputs,mode=SPIRAL_RECEIVE_MODEL)
receive_model_report=deepcopy(prepared["report"]["receive_model"])
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
assert np.allclose(prepared["combined_frame_matrix"],O,rtol=0,atol=1e-12)
assert preparation_report["all_children_in_field_limits"]
table([(key,preparation_report[key]) for key in ["requested_parent_count","baseline_B0_parent_count",
    "retained_parent_count","subspin_count","excluded_missing_B0_amplitude_fraction",
    "excluded_ACDC_amplitude_fraction_of_baseline","amplitude_area_ratio","slice_thickness_mm"]])
print("Mirrors / rotation: no image-array interpolation.")
table([("Receive model",receive_model_report["mode"]),
       ("Signal receivers",receive_model_report["receiver_count"]),
       ("ACDC current outputs",len(CHANNEL_MAX_CURRENT_A)),
       ("Receiver encoding fitted to this spiral scan",receive_model_report["estimated_from_spiral_scan"])])
if effective_simulation_model=="finite_voxel":
    table([(key,preparation_report[key]) for key in ["inplane_voxel_enabled",
        "excluded_missing_B0_gradient_parent_count","excluded_missing_ACDC_gradient_or_corner_parent_count",
        "inplane_voxel_field_limit_corners_per_child","ACDC_corner_linearization_max_abs_error_Hz_per_A"]],
        ("In-plane integration","Value"))

fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
image_physical(axes[0],prepared["native_amplitude"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Object in coil coordinates")
image_physical(axes[1],prepared["native_B0_Hz"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "B0 in coil coordinates [Hz]",cmap="RdBu_r")
image_physical(axes[2],prepared["parent_valid_mask"],prepared["physical_X_m"],prepared["physical_Y_m"],
               "Thickness-subspin coverage")
for column,color,label in [(0,"tab:red","+X native data"),(1,"tab:blue","+Y native data")]:
    axes[0].arrow(0,0,25*Q[0,column],25*Q[1,column],color=color,width=.8,length_includes_head=True)
    axes[0].text(30*Q[0,column],30*Q[1,column],label,color=color,fontsize=8)
show(fig)

uncalibrated_prepared=deepcopy(prepared)


## 4. Build the MRzero spiral model

The transverse amplitude is anchored at the first measured MEGRE TE, then
propagated to each spiral ADC. Native-to-output pixel-area and thickness weights
are applied once. Coil fields vary in 3D; a 2D B0 map remains constant through
the slice. The zero-B0 simulation retains the same object and relaxation and
is used only as a diagnostic.

With complex receiver encoding, the model first sums the weighted complex
subspin contributions within each receiver. It reconstructs all complex
receiver images independently and only then forms their RSS magnitude.
The receiver weights preserve the independently fitted spatial phases,
including the correction from the MEGRE reference echo back to excitation.
The historical RSS proxy remains available as an explicit comparison mode.

The model sums complex subspin contributions before taking any magnitude.
Finite-voxel integration includes the measured in-plane B0 and generated ACDC
gradients. The selected unresolved coherence factor is applied once and remains
independent of the currents. No unmeasured through-slice B0 distribution is inferred.

Command states follow spiral revolutions and have variable sample counts.
The amplifier response and accumulated phase are continuous between turns.
The fixed spiral reconstruction operator and its exact adjoint are used for
all forward images, image gradients and measured baseline comparisons.


In [ ]:
def reconstruction_factory(acquisition,device,dtype):
    return FrameReconstruction(reconstruction_acquisition(acquisition),device,dtype,frame_matrix=O,
                               sample_chunk=RECONSTRUCTION_CHUNK)


prepared=deepcopy(uncalibrated_prepared)
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
receive_model_report=deepcopy(preparation_report["receive_model"])
if "model" in globals():
    import gc
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

model_class=(SpiralReceiverACDCModel if receive_model_report["mode"]=="complex_echo" else
             MultiEchoVoxelACDCModel if complex_echo_enabled else
             FiniteVoxelEchoACDCModel if effective_simulation_model=="finite_voxel" else EchoSubspinACDCModel)
model_options=({"receive_chunk":RECEIVER_CHUNK} if model_class is SpiralReceiverACDCModel else {})
def build_spiral_model(prepared_inputs,sample_chunk=None):
    return model_class(prepared_inputs["expanded_experiment"],prepared_inputs["expanded_calibration"],
        max_current_A=CHANNEL_MAX_CURRENT_A,hardware=HARDWARE,
        reconstruction_factory=reconstruction_factory,profile=GPU_PROFILE,precision=GPU_PRECISION,
        memory_fraction=GPU_MEMORY_FRACTION,sample_chunk=sample_chunk,
        require_cuda=GPU_REQUIRED,**model_options)

model=build_spiral_model(prepared,sample_chunk=SAMPLE_CHUNK)
table([("Device / precision",f"{model.device} / {model.runtime_info['precision']}"),
       ("PyTorch / runtime CUDA",f"{torch.__version__} / {torch.version.cuda}"),
       ("Detected GPU",model.runtime_info["detected_gpu_name"])])
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
measured_coils=model.reconstruction.numpy_image(
    np.moveaxis(inputs["scan_diagnostics"]["corrected_kspace"],1,0))
measured_image=np.sqrt(np.sum(abs(measured_coils)**2,axis=0))
zero_currents=np.zeros((model.nc,model.nstates))
ideal_image=model.ideal_image
baseline_image,baseline_kdata=model.forward(zero_currents)
# Calibrate forward encoding while keeping the measured-data reconstruction fixed.
nominal_prepared=prepared
nominal_receiver_baseline_image=baseline_image.copy()
trajectory_parameters={"delay_x_s":0.,"delay_y_s":0.}
trajectory_calibration_report={"accepted":False,"parameters":trajectory_parameters.copy(),
    "reason":"Gradient-delay calibration disabled or complex receivers unavailable."}
if SPIRAL_GRADIENT_DELAY_CALIBRATION and isinstance(model,SpiralReceiverACDCModel):
    trajectory_result=fit_spiral_gradient_delays(model,inputs["acquisition"],
        inputs["paths"]["scan_sequence"],measured_image,O,
        max_delay_s=SPIRAL_MAX_GRADIENT_DELAY_US*1e-6)
    trajectory_parameters=trajectory_result["parameters"].copy()
    trajectory_calibration_report=deepcopy(trajectory_result["report"])
    if trajectory_result["accepted"]:
        prepared=apply_spiral_encoding(prepared,trajectory_parameters,
                                      sequence_path=inputs["paths"]["scan_sequence"])
        selected_chunk=int(model.sample_chunk)
        del model
        import gc
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
        model=build_spiral_model(prepared,sample_chunk=selected_chunk)
        baseline_image,baseline_kdata=model.forward(zero_currents)
        expected_rss=trajectory_result["candidate_rss"]
        trajectory_native_parity=float(np.linalg.norm(abs(baseline_image)-expected_rss)
            /max(np.linalg.norm(expected_rss),np.finfo(float).tiny))
        trajectory_calibration_report["rebuilt_native_RSS_relative_error"]=trajectory_native_parity
        if trajectory_native_parity>=1e-4:
            # Never apply a shortcut that disagrees with rebuilt native MRzero.
            trajectory_calibration_report.update(accepted=False,
                reason="Native rebuild disagreed with delayed-encoding candidate; nominal encoding restored.")
            trajectory_parameters={"delay_x_s":0.,"delay_y_s":0.}
            trajectory_calibration_report["parameters"]=trajectory_parameters.copy()
            prepared=nominal_prepared
            del model
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
            model=build_spiral_model(prepared,sample_chunk=selected_chunk)
            baseline_image,baseline_kdata=model.forward(zero_currents)
    del trajectory_result
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
receive_model_report=deepcopy(preparation_report["receive_model"])
ideal_image=model.ideal_image
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
uncalibrated_prepared=deepcopy(prepared)
model.runtime_info["spiral_gradient_delay_calibration"]=deepcopy(trajectory_calibration_report)
table([(key,value*1e6) for key,value in trajectory_parameters.items()],
      ("Effective native gradient delay","Applied [microseconds]"))
print(trajectory_calibration_report["reason"])
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
table([("Simulated spins",int(model.support.sum())),("MRzero cache storage",model.runtime_info["native_baseline_cache"]),
       ("Diagnostic MSE: no ACDC / zero-B0 simulation",baseline_model_mse),
       ("Twin / measured spiral NRMSE [%]",baseline_measured_metrics["NRMSE_percent"])])


point_baseline_image=None
point_baseline_metrics=None
fidelity_comparison=None
if COMPARE_POINT_BASELINE and effective_simulation_model=="finite_voxel":
    legacy_b0=np.asarray(inputs["field"]["b0Hz"])
    if B0_MODE=="regularized":
        legacy_b0=regularize_b0_local_plane(legacy_b0,inputs["field"]["mask"],
                                           sigma_pixels=B0_SIGMA_PIXELS)["b0Hz"]

    point_amplitude=prepared["native_amplitude"]
    point_rate=prepared["native_rate_sInv"]
    if complex_echo_enabled:
        if RATE_MODE=="guarded":
            point_amplitude=legacy_echo_fit["simulation_amplitude_at_reference"]
            point_rate=legacy_echo_fit["simulation_R2star_sInv"]
        else:
            point_amplitude=legacy_echo_fit["first_echo_amplitude"]
            pooled_rate=(legacy_echo_fit["pooled"].get("R2star_sInv",0.)
                         if RATE_MODE=="pooled" and legacy_echo_fit["pooled"].get("target_reliable_mask",False) else 0.)
            point_rate=np.full(point_amplitude.shape,pooled_rate)
    point_check=NativePointTrajectoryTwin(experiment["acquisition"],prepared["frame_X_m"],prepared["frame_Y_m"],
        point_amplitude*preparation_report["amplitude_area_ratio"],legacy_b0,
        prepared["parent_valid_mask"],rate_sInv=point_rate,
        amplitude_reference_time_s=fit["amplitude_reference_time_s"],device=str(model.device),sample_chunk=256)
    point_result=point_check.simulate(b0_enabled=True)
    point_baseline_image=model.reconstruction.numpy_image(point_result["kdata"][0])
    point_baseline_metrics=compare_magnitude(measured_image,point_baseline_image,X_m=X_image,Y_m=Y_image)
    bright_roi=measured_image>.2*measured_image.max()
    def low_signal_count(image,metrics,fraction):
        return int(np.count_nonzero(bright_roi & (metrics["gain"]*np.abs(image)<fraction*measured_image)))
    fidelity_comparison={"comparison":"same retained native parents; full-FOV evaluation gain only",
        "legacy_counter_direction":"simulation lower than measured spiral only; measured-dark regions excluded",
        "bright_ROI_fraction_of_measured_peak":.2,"bright_ROI_pixels":int(bright_roi.sum()),
        "point_NRMSE_percent":point_baseline_metrics["NRMSE_percent"],
        "revised_NRMSE_percent":baseline_measured_metrics["NRMSE_percent"],
        "point_below_half_measured_pixels":low_signal_count(point_baseline_image,point_baseline_metrics,.5),
        "revised_below_half_measured_pixels":low_signal_count(baseline_image,baseline_measured_metrics,.5)}
    fidelity_labels={"point_below_half_measured_pixels":"Point model: simulation < 50% of bright measurement [pixels]",
        "revised_below_half_measured_pixels":"Finite voxels: simulation < 50% of bright measurement [pixels]"}
    table([(fidelity_labels.get(key,key),value) for key,value in fidelity_comparison.items()],
          ("Legacy counter: simulated deficits only","Value"))
    panels=[baseline_measured_metrics["gain"]*np.abs(ideal_image),
            point_baseline_metrics["gain"]*np.abs(point_baseline_image),
            baseline_measured_metrics["gain"]*np.abs(baseline_image),measured_image]
    titles=["Spiral diagnostic: zero B0","Point model: historical RSS object",("Finite voxels + complex receivers" if receive_model_report["mode"]=="complex_echo" else
        "Finite voxels + 32-echo coherence" if complex_echo_enabled else "Finite voxels before calibration"),"Measured spiral without ACDC"]
    del point_check,point_result
else:
    gain=baseline_measured_metrics["gain"]
    panels=[gain*np.abs(ideal_image),gain*np.abs(baseline_image),measured_image]
    titles=["MRzero: zero B0","MRzero: measured B0","Measured spiral without ACDC (input)"]
fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),4),layout="constrained")
for ax,image,title in zip(axes,panels,titles):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)

historical_rss_baseline_image=None
historical_rss_baseline_metrics=None
receiver_model_comparison=None
if COMPARE_POINT_BASELINE and hasattr(model,"historical_rss_forward"):
    # Reuse native point histories; no second full MRzero cache is constructed.
    historical_rss_baseline_image,_=model.historical_rss_forward(zero_currents)
    historical_rss_baseline_metrics=compare_magnitude(
        measured_image,historical_rss_baseline_image,X_m=X_image,Y_m=Y_image)
    receiver_model_comparison={
        "stage":"Before baseline nuisance calibration; same retained object and temporal model",
        "historical_RSS_proxy_NRMSE_percent":historical_rss_baseline_metrics["NRMSE_percent"],
        "complex_receiver_RSS_NRMSE_percent":baseline_measured_metrics["NRMSE_percent"],
        "gain_policy":"A separate full-FOV scalar evaluation gain for each displayed model",
        "receiver_encoding_source":"Independent complex MEGRE measurement",
        "independent_spiral_quality_validation":False,
        "interpretation":"Descriptive comparison on the current spiral scan; not evidence of exact physical agreement"}
    table(list(receiver_model_comparison.items()),("Receiver model comparison","Value"))
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,image,title in zip(axes,[
            historical_rss_baseline_metrics["gain"]*abs(historical_rss_baseline_image),
            baseline_measured_metrics["gain"]*abs(baseline_image),measured_image],
            ["Historical RSS proxy: same finite voxels","Complex receivers, then image RSS","Measured spiral without ACDC"]):
        image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
    show(fig)

uncalibrated_baseline_image=baseline_image.copy()
uncalibrated_ideal_image=ideal_image.copy()
uncalibrated_baseline_kdata=baseline_kdata.copy()
uncalibrated_sample_chunk=int(model.sample_chunk)
uncalibrated_baseline_model_mse=baseline_model_mse
uncalibrated_baseline_measured_metrics=deepcopy(baseline_measured_metrics)


baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}


### 4a. Calibrate small baseline spiral discrepancies

`BASELINE_CALIBRATION='auto'` enables bounded calibration for measured inputs;
`'off'` disables it and `'on'` requests it explicitly. The object, relaxation,
spatial B0 variation and physical alignment remain fixed. Frequency and geometric
phase parameters are fitted only on fixed spatial fit blocks, selected on
separate validation blocks and checked on final test blocks. This is internal
validation of one acquisition, not an independent physical measurement.

Spiral data have no alternating Cartesian readout lines. An EPI odd/even phase
parameter is therefore inapplicable and is excluded from spiral calibration.
The calibration report describes the spiral parameters actually fitted.
With complex receive encoding, shared sample-phase candidates are applied
to every receiver, each receiver is reconstructed, and RSS and its exact
derivative are evaluated after reconstruction. Receiver weights remain fixed.
A solution is rejected if selection/test performance does not improve, bounds
are reached, or parameters are insufficiently identified.

Applied parameters rebuild the native MRzero model and resample the 3D coil maps
when required. Fixed-object and native-phase checks guard the rebuild. Re-running
this cell starts from the immutable uncalibrated inputs. No currents or measured
data are modified. Re-run subsequent diagnostics and optimization after calibration.


In [ ]:
import gc
import importlib
import spiral_baseline_calibration as baseline_calibration_tools
import spiral_baseline_calibration as baseline_audit_tools
baseline_calibration_tools=importlib.reload(baseline_calibration_tools)
baseline_audit_tools=importlib.reload(baseline_audit_tools)
if BASELINE_CALIBRATION not in {"auto","off","on"}:
    raise ValueError("BASELINE_CALIBRATION must be auto, off or on.")
baseline_calibration_enabled=(BASELINE_CALIBRATION=="on" or
    (BASELINE_CALIBRATION=="auto" and not is_point_synthetic))
baseline_metadata_audit=baseline_audit_tools.audit_baseline_inputs(inputs)
baseline_calibration_result=None
baseline_calibration_masks={name:np.zeros(measured_image.shape,dtype=bool)
                            for name in ("fit_mask","validation_mask","test_mask")}
baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
def prepare_baseline_candidate(parameters):
    rebuilt=prepare_echo_shim_inputs(inputs,fit,CALIBRATION_PATH,CALIBRATION_PARAMETERS,
        **angles,subspins_z=SUBSPINS_Z,rate_mode=effective_rate_mode,b0_mode=B0_MODE,
        b0_sigma_pixels=B0_SIGMA_PIXELS,current_response=CURRENT_RESPONSE,
        b0_weighting=effective_b0_weighting,inplane_voxel=effective_simulation_model=="finite_voxel",
        baseline_calibration=parameters)
    rebuilt=prepare_spiral_receive_model(rebuilt,fit,inputs,mode=SPIRAL_RECEIVE_MODEL)
    return apply_spiral_encoding(rebuilt,trajectory_parameters,
                                 sequence_path=inputs["paths"]["scan_sequence"])

def fixed_object_rebuild_checks(candidate):
    return fixed_spiral_object_checks(candidate,uncalibrated_prepared)

def validate_baseline_candidate(parameters):
    # Geometry checks do not inspect held-out scan pixels or run native MRzero.
    try:
        candidate=prepare_baseline_candidate(parameters)
        checks=fixed_object_rebuild_checks(candidate)
        return {"accepted":all(checks.values()),"checks":checks,
            "reason":"Object, quadrature and receive encoding must remain fixed after physical resampling."}
    except ValueError as error:
        return {"accepted":False,"reason":str(error)}

calibration_phase_acquisition=calibration_acquisition(
    inputs["acquisition"],experiment["acquisition"],O)

if baseline_calibration_enabled:
    baseline_calibration_result=baseline_calibration_tools.fit_baseline_calibration(
        calibration_phase_acquisition,uncalibrated_baseline_kdata,measured_image,
        reconstruction=reconstruction_factory(experiment["acquisition"],"cpu",torch.float64),
        bounds=CALIBRATION_BOUNDS,candidate_validator=validate_baseline_candidate)
    baseline_calibration_report=deepcopy(baseline_calibration_result["report"])
    baseline_calibration_parameters=baseline_calibration_result["parameters"].copy()
    for name in baseline_calibration_masks:
        baseline_calibration_masks[name]=baseline_calibration_result[name].copy()
else:
    baseline_calibration_report={"accepted":False,"parameters":baseline_calibration_parameters.copy(),
        "reason":"Calibration disabled for this synthetic input." if is_point_synthetic
                 else "Calibration disabled by BASELINE_CALIBRATION=off.",
        "gain_applied_to_object":False,"independent_acquisition_validation":False}
baseline_calibration_report["requested_mode"]=BASELINE_CALIBRATION
baseline_calibration_report["source_sha256"]=dict(inputs["source_sha256"],
    calibration=uncalibrated_prepared["report"]["calibration_source_sha256"])

def rebuild_baseline_model(parameters):
    rebuilt=prepare_baseline_candidate(parameters)
    return rebuilt,build_spiral_model(rebuilt,sample_chunk=uncalibrated_sample_chunk)

baseline_calibration_report["native_model_reused_without_parameter_change"]=not any(baseline_calibration_parameters.values())
if any(baseline_calibration_parameters.values()):
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
    baseline_image,baseline_kdata=model.forward(np.zeros((model.nc,model.nstates)))
else:
    # Reuse the validated native model instead of allocating its large cache again.
    prepared=uncalibrated_prepared
    baseline_image=uncalibrated_baseline_image.copy()
    baseline_kdata=uncalibrated_baseline_kdata.copy()
zero_currents=np.zeros((model.nc,model.nstates))
expected_calibrated_kdata=(uncalibrated_baseline_kdata*
    baseline_calibration_tools.phase_factor(calibration_phase_acquisition,baseline_calibration_parameters))
baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-expected_calibrated_kdata)
    /max(np.linalg.norm(expected_calibrated_kdata),np.finfo(float).tiny))
baseline_fixed_object_checks=fixed_object_rebuild_checks(prepared)
baseline_calibration_report["fixed_object_rebuild_checks"]=baseline_fixed_object_checks.copy()
if baseline_calibration_native_phase_parity>=1e-4 or not all(baseline_fixed_object_checks.values()):
    baseline_calibration_report["phase_parity_application_rejected"]=True
    baseline_calibration_report["candidate_native_phase_parity"]=baseline_calibration_native_phase_parity
    baseline_calibration_report["parameters_before_rebuild_gate"]=baseline_calibration_parameters.copy()
    baseline_calibration_report["accepted"]=False
    baseline_calibration_report["reason"]="Native parity failed or support/amplitude/relaxation/weights changed after resampling; zero parameters restored."
    baseline_calibration_parameters={name:0. for name in baseline_calibration_tools.PARAMETER_NAMES}
    baseline_calibration_report["parameters"]=baseline_calibration_parameters.copy()
    if "baseline" in baseline_calibration_report:
        baseline_calibration_report["applied"]=deepcopy(baseline_calibration_report["baseline"])
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    prepared,model=rebuild_baseline_model(baseline_calibration_parameters)
    zero_currents=np.zeros((model.nc,model.nstates))
    baseline_image,baseline_kdata=model.forward(zero_currents)
    baseline_calibration_native_phase_parity=float(np.linalg.norm(baseline_kdata-uncalibrated_baseline_kdata)
        /max(np.linalg.norm(uncalibrated_baseline_kdata),np.finfo(float).tiny))
baseline_calibration_report["applied_fixed_object_rebuild_checks"]=fixed_object_rebuild_checks(prepared)
assert (baseline_calibration_native_phase_parity<1e-4 and
        all(baseline_calibration_report["applied_fixed_object_rebuild_checks"].values())),"The restored model no longer matches the original baseline; reload the inputs."
experiment=prepared["expanded_experiment"]
calibration=prepared["expanded_calibration"]
preparation_report=prepared["report"]
receive_model_report=deepcopy(preparation_report["receive_model"])
baseline_calibration_report["receive_model"]=deepcopy(receive_model_report)
baseline_calibration_report["gradient_delay_calibration"]=deepcopy(trajectory_calibration_report)
X_image,Y_image=model.reconstruction.X_m,model.reconstruction.Y_m
ideal_image=model.ideal_image
baseline_model_mse=float(np.sum((abs(baseline_image)-abs(ideal_image))**2)/np.sum(abs(ideal_image)**2))
baseline_measured_metrics=compare_magnitude(measured_image,baseline_image,X_m=X_image,Y_m=Y_image)
baseline_calibration_report["native_phase_parity"]=baseline_calibration_native_phase_parity
baseline_calibration_report["native_sample_chunk"]={"uncalibrated":uncalibrated_sample_chunk,
    "requested_rebuild":uncalibrated_sample_chunk,"effective_rebuild":int(model.sample_chunk)}
baseline_calibration_report["preparation"]=deepcopy(preparation_report)
model.runtime_info["spiral_gradient_delay_calibration"]=deepcopy(trajectory_calibration_report)
model.runtime_info["baseline_calibration"]=baseline_calibration_parameters.copy()
model.runtime_info["baseline_calibration_accepted"]=baseline_calibration_report["accepted"]
model.runtime_info["baseline_calibration_native_phase_parity"]=baseline_calibration_native_phase_parity
model.runtime_info["baseline_calibration_original_sample_chunk"]=uncalibrated_sample_chunk

solution=None
image_history=None
solver_ok=False
print(baseline_calibration_report["reason"])
table(list(baseline_calibration_parameters.items()),("Model parameter","Applied value"))
if "baseline" in baseline_calibration_report:
    table([(name,baseline_calibration_report["baseline"][name]["NRMSE_percent"],
            baseline_calibration_report["applied"][name]["NRMSE_percent"])
           for name in ("fit","validation","test","full_FOV")],
          ("Pixels / gain fitted only on fit blocks","NRMSE before [%]","NRMSE after [%]"))
    before_gain=baseline_calibration_report["baseline_gain_from_fit_pixels"]
    after_gain=(baseline_calibration_report["candidate_gain_from_fit_pixels"]
                if baseline_calibration_report["accepted"] else before_gain)
else:
    before_gain=uncalibrated_baseline_measured_metrics["gain"]
    after_gain=before_gain
table([("Native MRzero / fast-phase parity",baseline_calibration_native_phase_parity),
       ("Diagnostic baseline / zero-B0 MSE",baseline_model_mse)])
print("Internal validation of one acquisition: check parameters on another scan before interpreting their physical origin.")
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
for ax,image,title in zip(axes,[measured_image,before_gain*np.abs(uncalibrated_baseline_image),
                               after_gain*np.abs(baseline_image)],
                         ["Measured spiral without ACDC","MRzero before calibration","MRzero after calibration"]):
    image_physical(ax,image,X_image,Y_image,title,vmin=0,vmax=measured_image.max()*1.1)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_calibration.npz",
    uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,
    uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    calibrated_baseline_image=baseline_image,calibrated_ideal_image=ideal_image,
    calibrated_baseline_kdata=baseline_kdata,measured_image=measured_image,
    receive_model_report_json=np.asarray(json.dumps(json_safe(receive_model_report),allow_nan=False)),
    **({"subspin_receive_weights":experiment["receive_weights"]} if "receive_weights" in experiment else {}),
    **baseline_calibration_masks)
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
(OUTPUT_DIR/"baseline_metadata_audit.json").write_text(
    json.dumps(json_safe(baseline_metadata_audit),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


baseline_signal_disagreement=None
baseline_signal_disagreement_arrays={}

(OUTPUT_DIR/"gradient_delay_calibration.json").write_text(
    json.dumps(json_safe(trajectory_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


### 4b. Check signal disagreement in both directions

A simulation can be too dark or too bright. Older counters only considered
measured-bright pixels and could miss real dark regions that were not reproduced.
This diagnostic reports both directions with the same fixed intensity scale:
red marks measured-dark/simulated-bright regions; blue marks the reverse.
Two low-intensity images are not automatically considered agreement.

The full FOV is the primary domain. A fixed envelope from the uncalibrated
zero-B0 diagnostic provides a secondary object-interior view; it changes no
spin support, field, reference image or objective. Gain comes only from baseline
fit pixels when calibration is active. Ratio thresholds of 25%/50% and the 10%
brightness threshold are descriptive; continuous signed error remains essential.
These maps do not by themselves identify irreversible signal loss or a B0 gradient
through the slice. They test baseline fidelity before fitting ACDC currents.


In [ ]:

import signal_disagreement as signal_disagreement_tools
signal_disagreement_tools=importlib.reload(signal_disagreement_tools)
signal_reference_scale=float(measured_image.max())
signal_envelope=signal_disagreement_tools.fixed_comparison_envelope(
    np.abs(uncalibrated_ideal_image),threshold_fraction=.1)
signal_subsets={name.removesuffix("_mask"):mask for name,mask in baseline_calibration_masks.items()
                if np.any(mask)}
signal_sources={"before_calibration":(uncalibrated_baseline_image,before_gain),
                "after_calibration":(baseline_image,after_gain)}
if point_baseline_image is not None:
    signal_sources={"point_before_calibration":(point_baseline_image,before_gain),**signal_sources}
baseline_signal_disagreement={"comparison_policy":{
    "primary_domain":"full FOV, including measured-dark pixels",
    "secondary_domain":"fixed envelope from uncalibrated zero-B0 target; diagnostic only",
    "envelope_threshold_fraction":.1,"reference_scale":signal_reference_scale,
    "bright_fraction":.1,"both_low_does_not_imply_agreement":True,
    "gain_source":"calibration fit pixels only" if baseline_calibration_enabled else "original full-FOV evaluation gain",
    "point_comparison_gain":"same before-calibration gain as the uncalibrated finite-voxel model",
    "object_target_field_and_optimization_loss_unchanged":True,
    "independent_physical_validation":False},"models":{}}
baseline_signal_disagreement_arrays={"fixed_comparison_envelope_mask":signal_envelope.copy(),"X_image_m":X_image.copy(),"Y_image_m":Y_image.copy()}
signal_rows=[]
signal_energy_rows=[]
signal_after_full=None
for name,(simulated,comparison_gain) in signal_sources.items():
    baseline_signal_disagreement["models"][name]={}
    for domain,analysis_mask in (("full_FOV",None),("fixed_envelope",signal_envelope)):
        result=signal_disagreement_tools.signal_disagreement(measured_image,simulated,
            gain=comparison_gain,analysis_mask=analysis_mask,subsets=signal_subsets or None,
            reference_scale=signal_reference_scale,bright_fraction=.1,ratios=(.25,.5))
        baseline_signal_disagreement["models"][name][domain]=result["report"]
        for key,value in result["arrays"].items():
            baseline_signal_disagreement_arrays[name+"_"+domain+"_"+key]=value
        maps=result["arrays"]
        energy=result["report"]["analysis_mask"]
        signal_energy_rows.append((name,domain,energy["NRMSE_percent"],
            energy["overprediction_normalized_SSE"],energy["underprediction_normalized_SSE"],
            energy["both_low_fraction_of_SSE"]))
        signal_rows.append((name,domain,
            int(np.count_nonzero(maps["measured_low_simulated_high_50_mask"])),
            int(np.count_nonzero(maps["simulated_low_measured_high_50_mask"])),
            int(np.count_nonzero(maps["both_low_mask"]))))
        if name=="after_calibration" and domain=="full_FOV":signal_after_full=result
table(signal_energy_rows,("Model","Domain","NRMSE [%]","Normalized SSE: simulation too bright",
    "Normalized SSE: simulation too dark","SSE fraction: both below threshold"))
table(signal_rows,("Model","Domain","Measured dark / simulation too bright [pixels]",
    "Simulation dark / measured bright [pixels]","Both low, agreement not implied [pixels]"))
print("Legacy counts measured only simulated deficits. Inspect measured losses that the model misses as well; colours do not identify their physical cause.")
print("Prioritize continuous signed error; descriptive ratios are 25% / 50%, with brightness >= 10% of the fixed measured maximum. Both images below threshold do not prove agreement.")
maps=signal_after_full["arrays"]
classification=np.full(measured_image.shape,np.nan)
classification[maps["simulated_low_measured_high_50_mask"]]=-1.
classification[maps["measured_low_simulated_high_50_mask"]]=1.
difference=maps["difference"]
diff_max=max(float(np.nanmax(np.abs(difference))),np.finfo(float).tiny)
fig,axes=plt.subplots(1,4,figsize=(20,4),layout="constrained")
image_physical(axes[0],measured_image,X_image,Y_image,"Measured spiral without ACDC",vmin=0,vmax=signal_reference_scale)
image_physical(axes[1],after_gain*np.abs(baseline_image),X_image,Y_image,"Compared MRzero baseline",vmin=0,vmax=signal_reference_scale)
im=image_physical(axes[2],difference,X_image,Y_image,"Simulation − measurement\nRed: simulation too bright",cmap="RdBu_r",vmin=-diff_max,vmax=diff_max)
fig.colorbar(im,ax=axes[2],shrink=.8)
image_physical(axes[3],measured_image,X_image,Y_image,"Red: measurement below simulation\nBlue: simulation below measurement (50% ratio, 10% brightness)",vmin=0,vmax=signal_reference_scale)
axes[3].pcolormesh(X_image*1000,Y_image*1000,classification,shading="auto",cmap="RdBu_r",vmin=-1,vmax=1,alpha=.8)
show(fig)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(OUTPUT_DIR/"baseline_signal_disagreement.npz",**baseline_signal_disagreement_arrays)
(OUTPUT_DIR/"baseline_signal_disagreement.json").write_text(
    json.dumps(json_safe(baseline_signal_disagreement),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
baseline_calibration_report["native_model_signal_disagreement"]=baseline_signal_disagreement
(OUTPUT_DIR/"baseline_calibration.json").write_text(
    json.dumps(json_safe(baseline_calibration_report),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


### Validate the signal model before optimization

For finite voxels, compare analytic in-plane integration with independent numeric
quadrature. For point spins, compare the zero-current MRzero signal with the
independent point-spin twin on the same retained parents. A point reference is
not a physical oracle for a finite-voxel model. Native MRzero resummation checks
remain active, and field constraints still include finite-voxel corners.


In [ ]:
if effective_simulation_model=="finite_voxel":
    reference_validation_report=model.validate_voxel_quadrature()
    zero_current_parity=None
    table(list(reference_validation_report.items()),("Voxel quadrature check","Value"))
    if not reference_validation_report["passed"]:
        raise AssertionError("Analytic voxel integration does not agree with independent quadrature.")
else:
    reference_check=NativePointTrajectoryTwin(experiment["acquisition"],prepared["frame_X_m"],prepared["frame_Y_m"],
        prepared["native_amplitude"]*preparation_report["amplitude_area_ratio"],prepared["native_B0_Hz"],
        prepared["parent_valid_mask"],rate_sInv=prepared["native_rate_sInv"],
        amplitude_reference_time_s=fit["amplitude_reference_time_s"],device=str(model.device),sample_chunk=128)

    reference_result=reference_check.simulate(b0_enabled=True)
    reference_validation_report=reference_result["report"]
    zero_current_parity=float(np.linalg.norm(reference_result["kdata"][0]-baseline_kdata)/np.linalg.norm(baseline_kdata))
    assert np.isfinite(zero_current_parity) and zero_current_parity<1e-4, (
        f"Point-twin / complete-model zero-current mismatch: {zero_current_parity:.3g} (threshold 1e-4).")
    table([("MRzero resummation: relative signal error",reference_validation_report["native_resummation_relative_error"]),
           ("MRzero resummation: error / tolerance (≤1)",reference_validation_report["native_resummation_max_tolerance_ratio"]),
           ("Point-twin / complete-model zero-current relative error",zero_current_parity)])
    del reference_check,reference_result
    if model.device.type=="cuda":torch.cuda.empty_cache()


## 5. Static-shim initialization and hardware checks

The constrained static field solution initializes all spiral-turn commands.
The hardware projection then checks the actual temporal response, full-TR
limits and all retained subspins. Disabled physical channels remain exactly zero.
Unmeasured electrical and thermal parameters remain unset rather than invented.


In [ ]:
if IMAGE_INITIALIZATION=="static":
    static_fit=solve_field_gpu(model.maps_numpy[model.quality_mask],
        -experiment["field"]["b0Hz"][model.quality_mask],device=model.device,dtype=model.dtype,
        max_current_A=CHANNEL_MAX_CURRENT_A,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
        field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=1000,tolerance=1e-7)
    requested_initial=np.repeat(static_fit["currents_A"][:,:1],model.nstates,axis=1)
elif IMAGE_INITIALIZATION=="zero":requested_initial=zero_currents.copy()
else:raise ValueError("IMAGE_INITIALIZATION must be static or zero.")
initial_currents=model.project_currents(requested_initial,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
initial_hardware=model.hardware.diagnostics(initial_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
assert initial_hardware["all_limits_met"]
initial_image,_=model.forward(initial_currents)
table(list(initial_hardware["checks"].items()),("Initialization check","Result"))
print("Equivalent response: tau [µs] =",model.current_response["tau_s"][:3]*1e6)


## 6. Optimize one 32-channel current vector per spiral turn

The image objective uses the **selected measured GRE**, interpolated into the
spiral image grid in physical coordinates. The comparison mask and positive gain
are fixed before optimization and retained after DAC quantization:

`J_image(I) = sum_M((g * abs(spiral_MRzero(I)) - GRE_measured)**2) / sum_M(GRE_measured**2)`.

The signal model uses B0, the 32-echo temporal model, 3D ACDC fields and their exact
response at every ADC. It sums complex contributions before reconstructing the
magnitude image. Different turns can contain different numbers of ADC samples.
No reshape into Cartesian lines or FFT surrogate is used.

The live image history stores magnitudes only at iterations
`0, LIVE_UPDATE_EVERY, 2*LIVE_UPDATE_EVERY, ...`. Scalar losses are retained at
all evaluated iterations. `optimization_images.npz` and
`optimization_images.html` save the gallery at each update, with one fixed scale.
The final live snapshot can differ from the best retained solution.

When `FIELD_GUIDANCE_WEIGHT > 0`, the total objective is
`J_total = J_image + weight * mean((B0_residual / scale_Hz)**2)`.
The residual field is evaluated at physical reliable subspin centres using exact
integrals of the actual amplifier response between consecutive turn centres.
The guide uses the measured unwrapped B0 branch in Hz. It does not use a phase
modulo or infer an absolute frequency branch from an image.
`0.05` is an initial weight to compare experimentally; `0` gives image-only fitting.
`FIELD_GUIDANCE_SCALE_HZ=100` normalizes the penalty and is not a field limit.

Image MSE/NRMSE and total objective are reported separately. Selection uses the
total objective; `TARGET_MSE` remains diagnostic in guided mode. The guide requires
a correctly unwrapped and frequency-referenced B0 map with the same scanner
frequency as the spiral acquisition. It respects all channel, current and field
limits and does not guarantee recovery of attenuated signal.

The runtime report distinguishes compute device from MRzero-cache storage.
A CPU cache can feed actual GPU computation in chunks. `use_cuda_graph=False`
preserves hardware checks and does not disable CUDA computation.


In [ ]:
optimization_target=configure_measured_gre_target(model,inputs["gre"],
    baseline_image=baseline_image,measured_baseline_image=measured_image,
    frame_matrix=O,threshold_fraction=OPTIMIZATION_REFERENCE_THRESHOLD)
optimization_reference_image=optimization_target["reference_image"]
optimization_reference_mask=optimization_target["mask"]
optimization_target_report=optimization_target["report"]
table(list(optimization_target_report.items()),("Optimization GRE target","Value"))
table([("Baseline / measured GRE MSE",relative_image_mse(model,baseline_image)),
       ("Initialization / measured GRE MSE",relative_image_mse(model,initial_image))])
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
target_vmax=max(float(optimization_reference_image.max()),float(np.abs(baseline_image).max()))
for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,initial_image],
                         ["Target: measured GRE (fixed scale)","Simulated spiral without ACDC","Initial simulated spiral"]):
    image_physical(ax,np.where(optimization_reference_mask,np.abs(image),np.nan),
        X_image,Y_image,title,vmin=0,vmax=target_vmax)
show(fig)

solution=None
image_history=None
if RUN_OPTIMIZATION:
    if GPU_REQUIRED and model.device.type!="cuda":
        raise RuntimeError("Optimization requires CUDA but the current model is on CPU. "
            "Fix the PyTorch/CUDA environment, restart the kernel and rerun from the beginning. "
            "Changing GPU_PROFILE alone does not move an existing model.")
    table([("Optimization device",str(model.device)),
           ("Model tensors",str(model.B.device)),
           ("Precision",str(model.dtype)),
           ("GPU",model.runtime_info["gpu_name"]),
           ("MRzero cache storage",model.runtime_info["native_baseline_cache"]),
           ("ADC samples per chunk",model.sample_chunk)])
    image_history=OptimizationImageHistory(optimization_reference_image,baseline_image,every=LIVE_UPDATE_EVERY,
        title="Scan current — optimization image history",origin="upper",language="en")
    image_history_npz=OUTPUT_DIR/"optimization_images.npz"
    image_history_html=OUTPUT_DIR/"optimization_images.html"
    live=LiveOptimizationPlot(reference=optimization_reference_image,baseline=baseline_image,
        current_limit_a=MAX_CURRENT_A,roi_mask=optimization_reference_mask,
        title=f"07 spiral: alignment {ALIGNMENT_ANGLE_DEG}°, mirrors X={ALIGNMENT_MIRROR_X}, Y={ALIGNMENT_MIRROR_Y}")
    set_turn_axis_labels(live)
    def update(snapshot):
        live.set_history(snapshot["history"]["iteration"],snapshot["history"]["fval"])
        live.update(iteration=snapshot["iteration"],image=snapshot["image"],currents_a=snapshot["currents_A"],
            loss=snapshot["fval"],nrmse_percent=100*np.sqrt(snapshot["fval"]),
            max_field_hz=snapshot["max_field_Hz"],field_limit_hz=FIELD_LIMIT_HZ)
        if image_history.record(snapshot):
            image_history.save_npz(image_history_npz)
            image_history.write_html(image_history_html)
            image_history.display()
    try:
        solution=optimize_mrzero_image(model,initial_currents,total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ,max_iterations=MAX_ITERATIONS,learning_rate_A=LEARNING_RATE_A,
            target_mse=TARGET_MSE,patience=PATIENCE,callback=update,callback_every=LIVE_UPDATE_EVERY,
            use_cuda_graph=False,field_guidance_weight=FIELD_GUIDANCE_WEIGHT,
            field_guidance_scale_Hz=FIELD_GUIDANCE_SCALE_HZ)
    finally:
        live.finish("Finished" if solution is not None else "Stopped: last evaluated state")
    info=solution["optimization_info"]
    currents=solution["currents_A"]
    history=solution["history"]
    table(list(info.items()))
    if "field_guidance" in info:
        guide_report=info["field_guidance"]
        table([("Initial total objective",solution["initial_objective"]),("Selected total objective",info["objective"]),
               ("Initial residual B0 RMS [Hz]",guide_report["initial_residual_field_rms_Hz"]),
               ("Selected residual B0 RMS [Hz]",guide_report["final_residual_field_rms_Hz"])])
    print("Image gallery:",image_history_html)
else:print("Optimization disabled; model diagnostics remain available.")


## 7. Verify continuous currents and DAC quantization

Check hardware limits between ADC samples and across the full TR, then check them
again after nominal DAC conversion. Image MSE must improve on initialization.
With guidance active, the quantized total objective must also improve. The DAC
check uses the saved weight/scale from the actual optimization result.
An iteration limit is reported explicitly and is not a mathematical convergence claim.


In [ ]:
solver_ok=False
if solution is not None:
    hardware_report=model.hardware.diagnostics(currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_currents=quantize_currents(currents)
    quantized_hardware=model.hardware.diagnostics(quantized_currents,TOTAL_CURRENT_LIMIT_A,FIELD_LIMIT_HZ)
    quantized_image,quantized_kdata=model.forward(quantized_currents)
    quantized_mse=relative_image_mse(model,quantized_image)
    quantized_objective=quantized_mse
    quantized_field_guidance=None
    if info.get("field_guidance_weight",0.)>0:
        dac_guide=FieldGuidance(model,scale_Hz=info["field_guidance_scale_Hz"])
        dac_penalty,_,dac_rms=dac_guide.value_gradient(model._check_currents(quantized_currents),gradient=False)
        quantized_objective+=info["field_guidance_weight"]*float(dac_penalty)
        quantized_field_guidance={"field_penalty":float(dac_penalty),"residual_field_rms_Hz":float(dac_rms)}
        # Release the temporary guide so it cannot retain this model across notebook reruns.
        del dac_guide,dac_penalty,dac_rms
    solver_ok=bool(hardware_report["all_limits_met"] and quantized_hardware["all_limits_met"]
        and np.isfinite(info["fval"]) and info["fval"]<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_mse) and quantized_mse<=solution["initial_relative_mse"]+1e-7
        and np.isfinite(quantized_objective) and quantized_objective<=solution["initial_objective"]+1e-7)
    table([(key,value,quantized_hardware["checks"][key]) for key,value in hardware_report["checks"].items()],
          ("Hardware check","Optimized","After DAC"))
    table([("Initial MSE",solution["initial_relative_mse"]),("Optimized MSE",info["fval"]),
           ("After-DAC MSE",quantized_mse),("Numerical export checks",solver_ok)])
    if quantized_field_guidance is not None:
        table([("After-DAC total objective",quantized_objective),
               ("After-DAC residual B0 RMS [Hz]",quantized_field_guidance["residual_field_rms_Hz"])])
    fig,axes=plt.subplots(1,4,figsize=(18,4),layout="constrained")
    vmax=max(abs(image).max() for image in (optimization_reference_image,baseline_image,solution["image"],quantized_image))
    for ax,image,title in zip(axes,[optimization_reference_image,baseline_image,solution["image"],quantized_image],
                             ["Measured GRE target","Without ACDC","Optimized ACDC","After DAC"]):
        image_physical(ax,abs(image),X_image,Y_image,title,vmin=0,vmax=vmax)
    show(fig)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    im=axes[0].imshow(currents,cmap="RdBu_r",aspect="auto",vmin=-MAX_CURRENT_A,vmax=MAX_CURRENT_A)
    fig.colorbar(im,ax=axes[0]);axes[0].set(title="DAC commands",xlabel="Spiral turn",ylabel="Channel")
    axes[1].plot(history["iteration"],history["fval"],label="Evaluated")
    axes[1].plot(history["iteration"],history["best_fval"],label="Best image MSE evaluated")
    axes[1].set(title="Relative MSE",xlabel="Iteration");axes[1].legend()
    axes[2].bar(np.arange(model.nc)+1,hardware_report["avg_abs_current_A"])
    axes[2].set(title="Mean physical current over the TR",xlabel="Channel",ylabel="Mean |I| [A]")
    show(fig)
    if "objective" in history:
        fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")
        axes[0].plot(history["iteration"],history["objective"],label="Evaluated")
        axes[0].plot(history["iteration"],history["best_objective"],label="Best total objective")
        axes[0].set(title="Image + B0 guidance",xlabel="Iteration");axes[0].legend()
        axes[1].plot(history["iteration"],history["residual_field_rms_Hz"])
        axes[1].set(title="Physical residual field",xlabel="Iteration",ylabel="RMS [Hz]")
        show(fig)


## 8. Inspect amplifier response and physical subspin fields

Compare instantaneous and filtered predictions for exactly the same commands.
Displayed parent-field maps average children only for plotting. Complex phase,
current response and field limits were evaluated at individual subspins/corners.
Turn indices below refer to chronological command states, not Cartesian lines.


In [ ]:
if solution is not None:
    response=solution["response_diagnostics"]
    instantaneous_image,_=model.forward(currents,response_enabled=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,image,title in zip(axes,[instantaneous_image,solution["image"],abs(instantaneous_image)-abs(solution["image"])],
                             ["Same commands: instantaneous response","Filtered response: 133 µs settling","Magnitude difference"]):
        image_physical(ax,abs(image) if "difference" not in title.lower() else image,X_image,Y_image,title,
                       cmap="RdBu_r" if "difference" in title.lower() else "gray")
    show(fig)
    # Historical response key: these centres are spiral-turn centres in version 07.
    field_parent=response["generated_field_at_line_center_Hz"]
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,index in zip(axes,[0,model.nstates//2,model.nstates-1]):
        im=image_physical(ax,field_parent[...,index],prepared["physical_X_m"],prepared["physical_Y_m"],
                          f"Mean ACDC field: turn {index+1} [Hz]",cmap="RdBu_r")
        fig.colorbar(im,ax=ax,shrink=.8)
    show(fig)


## 9. Save the complete session and orientation

The archive includes Q/P/O, mirrors, native and patient-space subspin positions,
anchored amplitudes, B0, decay/coherence parameters, actual k/time coordinates,
turn boundaries, commands, hardware limits, DAC checks and input hashes.
Fixed complex receive weights, verified physical receiver IDs and receive-model
assumptions are saved alongside those arrays. Receive channels and ACDC
current outputs retain separate identities.
Unique session names preserve previous results. Periodic images have a separate
archive; the session saves initial/final currents without a full current history.


In [ ]:
saved_guidance_configuration=(
    {"weight":solution["optimization_info"]["field_guidance_weight"],
     "scale_Hz":solution["optimization_info"]["field_guidance_scale_Hz"]}
    if solution is not None else
    {"weight":FIELD_GUIDANCE_WEIGHT,"scale_Hz":FIELD_GUIDANCE_SCALE_HZ})
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
arrays={"echo_times_s":inputs["echo_times_s"],
    "object_support":fit["object_support"],"noise_mask":fit["noise_mask"],
    "noise_covariance":fit["noise_covariance"],
    "support_snr":fit["support_snr"],"support_debiased_amplitude":fit["support_debiased_amplitude"],
    "fit_valid_mask":fit["fit_valid_mask"],"target_reliable_mask":fit["target_reliable_mask"],
    "R2star_sInv":fit["R2star_sInv"],"R2star_std_sInv":fit["R2star_std_sInv"],
    "ideal_image":ideal_image,"baseline_image":baseline_image,"measured_image":measured_image,
    "initial_currents_A":initial_currents,"X_image_m":X_image,"Y_image_m":Y_image,
    "field_guidance_weight":np.asarray(saved_guidance_configuration["weight"]),
    "field_guidance_scale_Hz":np.asarray(saved_guidance_configuration["scale_Hz"]),
    "channel_max_current_A":CHANNEL_MAX_CURRENT_A,
    "channel_map":np.asarray(WAVEFORM_PARAMETERS["channel_map"],dtype=int),
    "disabled_physical_channels":np.asarray(WAVEFORM_PARAMETERS["disabled_channels"],dtype=int),
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "native_X_m":prepared["native_X_m"],"native_Y_m":prepared["native_Y_m"],
    "native_amplitude":prepared["native_amplitude"],"native_rate_sInv":prepared["native_rate_sInv"],
    "native_B0_Hz":prepared["native_B0_Hz"],"parent_valid_mask":prepared["parent_valid_mask"],
    "parent_flat_index":prepared["parent_flat_index"],"subspin_weights":prepared["weights"],
    "subspin_xyz_patient_m":prepared["xyz_m"],"subspin_B0_Hz":prepared["subspin_b0"],
    "subspin_basis_Hz_per_A":prepared["fieldbasis"],"subspin_amplitude_at_reference":experiment["m0_proxy"],
    "kx_mInv":experiment["acquisition"]["kx_mInv"],"ky_mInv":experiment["acquisition"]["ky_mInv"],
    "ADC_time_s":experiment["acquisition"]["time_s"]}
arrays.update(optimization_reference_image=optimization_reference_image,
    optimization_reference_mask=optimization_reference_mask,
    optimization_reference_measured_magnitude=optimization_target["measured_reference_magnitude"],
    optimization_reference_coverage_mask=optimization_target["coverage_mask"],
    optimization_fixed_gain=np.asarray(optimization_target["gain"]),
    measured_GRE_native_magnitude=inputs["gre"]["magnitude"],
    measured_GRE_native_X_m=inputs["gre"]["X_m"],measured_GRE_native_Y_m=inputs["gre"]["Y_m"],
    zero_b0_diagnostic_image=ideal_image)
arrays.update(uncalibrated_baseline_image=uncalibrated_baseline_image,
    uncalibrated_ideal_image=uncalibrated_ideal_image,uncalibrated_baseline_kdata=uncalibrated_baseline_kdata,
    original_native_X_m=prepared["original_native_X_m"],original_native_Y_m=prepared["original_native_Y_m"],
    native_measured_B0_Hz=prepared["native_measured_B0_Hz"],
    ADC_phase_offset_rad=experiment["acquisition"].get("phase_offset_rad",np.zeros_like(experiment["acquisition"]["time_s"])),
    **{"baseline_calibration_"+key:value for key,value in baseline_calibration_masks.items()})
arrays.update({"signal_disagreement_"+key:value for key,value in baseline_signal_disagreement_arrays.items()})
arrays["receive_model_report_json"]=np.asarray(json.dumps(json_safe(receive_model_report),allow_nan=False))
if "receive_weights" in experiment:
    arrays["subspin_receive_weights"]=experiment["receive_weights"]
for label,metadata in (("spiral",inputs["scan_metadata"]),("MEGRE",inputs["b0_raw"]["metadata"])):
    if "receiver_channel_ids" in metadata:
        arrays[label+"_receiver_channel_ids"]=np.asarray(metadata["receiver_channel_ids"],dtype=int)
if historical_rss_baseline_image is not None:
    arrays["historical_rss_baseline_image"]=historical_rss_baseline_image
if point_baseline_image is not None:
    arrays["point_baseline_image"]=point_baseline_image
if effective_simulation_model=="finite_voxel":
    for name in ("edge_vectors_m","b0_gradient_Hz_per_pixel","field_gradient_Hz_per_A_per_pixel"):
        arrays["inplane_"+name]=experiment["inplane_voxel"][name]
if solution is not None:
    arrays.update(optimized_image=solution["image"],optimized_kdata=solution["kdata"],currents_A=currents,
        quantized_currents_A=quantized_currents,quantized_image=quantized_image,
        iteration=history["iteration"],relative_mse=history["fval"],best_relative_mse=history["best_fval"])
    for key in ("objective","best_objective","field_penalty","residual_field_rms_Hz"):
        if key in history:arrays["optimization_"+key]=history[key]
if complex_echo_enabled:
    ce=fit["multi_echo"]
    arrays.update(complex_echo_width_Hz=ce["unresolved_width_Hz"],
        complex_echo_dispersion_kind=ce["dispersion_kind"],
        complex_echo_residual_R2_sInv=ce["residual_R2_sInv"],
        complex_echo_envelope_at_reference=ce["amplitude_envelope_at_reference"],
        complex_echo_refined_B0_Hz=ce["refined_b0_Hz"],
        complex_echo_predicted_RSS=ce["predicted_RSS"],
        complex_echo_hole_mask=complex_echo_diagnostics["hole_mask"],
        complex_echo_known_gradient_Hz_per_pixel=ce["known_inplane_gradient_Hz_per_pixel"],
        complex_echo_support_mask=ce["support_mask"],complex_echo_quality_mask=ce["quality_mask"],
        complex_echo_reduced_chi2=ce["reduced_chi2"],
        complex_echo_training_relative_RMSE=ce["training_relative_complex_RMSE"],
        complex_echo_validation_relative_RMSE=ce["validation_relative_complex_RMSE"],
        complex_echo_receiver_amplitude=ce["complex_receiver_amplitude"])
if complex_echo_enabled:
    for key in ("reference_amplitude_ratio","reference_amplitude_coverage_mask",
                "reference_contrast_field","reference_contrast_coverage_mask"):
        if key in fit["multi_echo"]:
            arrays[key]=fit["multi_echo"][key]
# Store the original sample order and actual variable-duration turn boundaries.
for name in ("line_index","state_index","turn_index","turn_start_s","turn_end_s","state_start_s","state_end_s"):
    if name in experiment["acquisition"]:
        arrays["spiral_"+name]=experiment["acquisition"][name]
arrays["density_compensation_weights"]=np.asarray(inputs["acquisition"].get("density_compensation_weights",np.ones_like(inputs["acquisition"]["time_s"])))
arrays["command_start_s"]=np.asarray(model.line_start_s)
arrays["command_end_s"]=np.asarray(model.line_end_s)
arrays["nominal_receiver_baseline_image"]=nominal_receiver_baseline_image
arrays["forward_kx_mInv"]=experiment["acquisition"]["kx_mInv"]
arrays["forward_ky_mInv"]=experiment["acquisition"]["ky_mInv"]
arrays["nominal_reconstruction_kx_mInv"]=reconstruction_acquisition(experiment["acquisition"])["kx_mInv"]
arrays["nominal_reconstruction_ky_mInv"]=reconstruction_acquisition(experiment["acquisition"])["ky_mInv"]
arrays["gradient_delay_parameters_s"]=np.asarray([trajectory_parameters[name] for name in ("delay_x_s","delay_y_s")])
np.savez_compressed(OUTPUT_DIR/"mrzero07_optimization.npz",**arrays)
source_hashes=dict(inputs["source_sha256"],calibration=preparation_report["calibration_source_sha256"])
summary={"notebook":"07_acdc_spiral_echo32_b0_s0_t2star_MRzero.ipynb",
    "sequence_assessment":sequence_assessment,
    "reconstruction_configuration":{"method":SPIRAL_RECONSTRUCTION_METHOD,"density_compensation":SPIRAL_DENSITY_COMPENSATION},
    "reconstruction_report":model.reconstruction.report,
    "receive_model_configuration":{"requested_mode":SPIRAL_RECEIVE_MODEL,"receiver_chunk":RECEIVER_CHUNK},
    "receive_model":receive_model_report,
    "receiver_model_comparison":receiver_model_comparison,
    "gradient_delay_calibration":trajectory_calibration_report,
    "gradient_delay_configuration":{"enabled":SPIRAL_GRADIENT_DELAY_CALIBRATION,"bound_us":SPIRAL_MAX_GRADIENT_DELAY_US},
    "trajectory_kind":"spiral", "command_state_semantics":"one 32-channel vector per verified spiral turn",
    "input_selection":{name:str(path.resolve()) for name,path in inputs["paths"].items()},
    "reference_sequence_validation":inputs.get("reference_sequence_validation"),
    "reference_geometry_checks":inputs.get("geometry_checks"),
    "reference_amplitude_prior":fit["report"].get("reference_amplitude_prior"),
    "optimization_target":optimization_target_report,
    "field_guidance_configuration":saved_guidance_configuration,
    "zero_b0_image_role":"Diagnostic only; optimization target is the selected measured GRE.",
    "multi_echo_fit":fit["report"].get("multi_echo"),
    "multi_echo_comparison":None if complex_echo_diagnostics is None else complex_echo_diagnostics["rows"],
    "echo_input_configuration":{"B0_sequence":B0_SEQUENCE_NAME,"B0_scan":inputs["paths"]["b0"],
        "field_echo_count":inputs["field_echo_count"],"rate_mode":effective_rate_mode,"echo_signal_model":ECHO_SIGNAL_MODEL,
        "object_amplitude_source":OBJECT_AMPLITUDE_SOURCE,
        "object_contrast_sigma_mm":OBJECT_CONTRAST_SIGMA_MM,
        "optimization_reference_threshold":OPTIMIZATION_REFERENCE_THRESHOLD,
        "reference_prior_min_validation_improvement":REFERENCE_PRIOR_MIN_VALIDATION_IMPROVEMENT,
        "object_threshold_fraction":OBJECT_THRESHOLD_FRACTION,"object_min_snr":OBJECT_MIN_SNR,
        "background_corner_m":BACKGROUND_CORNER_M,"require_noise_calibration":REQUIRE_NOISE_CALIBRATION},
    "input_dir":INPUT_DIR,"calibration_path":CALIBRATION_PATH.resolve(),
    "waveform_output_dir":WAVEFORM_OUTPUT_DIR.resolve(),"baseline_scan":inputs["paths"]["scan"].name,"orientation":angles,
    "alignment_matrix":Q,"passive_frame_matrix":P,"combined_frame_matrix":O,
    "preparation":preparation_report,"echo_fit":fit["report"],"runtime":model.runtime_info,
    "fidelity_configuration":{"requested_model":SIMULATION_MODEL,"effective_model":effective_simulation_model,
        "requested_B0_weighting":B0_WEIGHTING,"effective_B0_weighting":effective_b0_weighting},
    "baseline_signal_disagreement":baseline_signal_disagreement,
    "baseline_calibration":baseline_calibration_report,"baseline_metadata_audit":baseline_metadata_audit,
    "uncalibrated_baseline_measured_comparison":uncalibrated_baseline_measured_metrics,
    "uncalibrated_baseline_vs_zeroB0_mse":uncalibrated_baseline_model_mse,
    "fidelity_comparison_stage":"before baseline calibration",
    "fidelity_comparison":fidelity_comparison,"point_baseline_measured_comparison":point_baseline_metrics,
    "zero_current_validation":reference_validation_report,
    "zero_current_reference":reference_validation_report,
    "zero_current_V4_parity":zero_current_parity,"baseline_measured_comparison":baseline_measured_metrics,
    "baseline_vs_zeroB0_mse":baseline_model_mse,"current_limits_A":{"per_channel":CHANNEL_MAX_CURRENT_A,"total":TOTAL_CURRENT_LIMIT_A},
    "current_response":CURRENT_RESPONSE,"hardware_configuration":HARDWARE,
    "disabled_channels":WAVEFORM_PARAMETERS["disabled_channels"],
    "waveform_parameters":WAVEFORM_PARAMETERS,
    "source_sha256":source_hashes,"export_checks_passed":solver_ok,
    "image_history":{"every":LIVE_UPDATE_EVERY,
        "iterations":image_history.arrays()["iterations"] if image_history is not None else [],
        "archive":"optimization_images.npz" if image_history is not None else None,
        "viewer":"optimization_images.html" if image_history is not None else None}}
summary["waveform_adc_export"]=globals().get("adc_waveform_export")
if solution is not None:
    summary.update(optimization=info,hardware=hardware_report,hardware_after_DAC=quantized_hardware,
        quantized_relative_mse=quantized_mse,initial_relative_mse=solution["initial_relative_mse"],
        initial_objective=solution["initial_objective"],quantized_objective=quantized_objective,
        quantized_field_guidance=quantized_field_guidance)
(OUTPUT_DIR/"mrzero07_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("Session saved:",OUTPUT_DIR)


### 9a. Export independent static and dynamic field shims

Both field-fitting methods use the same calibrated maps, disabled channels,
physical frame, response model, hardware constraints and synchronization as the
image solution. They run independently of `RUN_OPTIMIZATION` and
`IMAGE_INITIALIZATION`. Dynamic field commands are fitted on the same spiral-turn
state boundaries. CSV files and companion checks are retained under
`INPUT_DIR/waveform`; numbering preserves previous exports.


In [ ]:
field_shim_exports=None
if EXPORT_FIELD_SHIM_WAVEFORMS:
    field_shim_exports=export_field_shim_waveforms(
        model,WAVEFORM_OUTPUT_DIR,WAVEFORM_PARAMETERS,
        total_current_limit_A=TOTAL_CURRENT_LIMIT_A,field_limit_Hz=FIELD_LIMIT_HZ,
        timing=inputs["timing"],sequence_path=inputs["paths"]["scan_sequence"],
        csv_time_origin=CSV_TIME_ORIGIN,trigger_channel=TRIGGER_CHANNEL,
        raster_s=FIELD_SHIM_RASTER_S,max_iterations=FIELD_SHIM_MAX_ITERATIONS,
        provenance=globals().get("source_hashes",{}))
    summary["field_shim_waveforms"]=field_shim_exports
    table([(name,entry["csv_file"]) for name,entry in field_shim_exports["exports"].items()],
          ("Field-shim method","Saved CSV"))
else:
    print("Static and dynamic exports disabled.")


## 10. Export synchronized current CSV files

The actual Pulseq `ext1` turn triggers are checked before export. In
`first_imaging_trigger` mode, the first verified trigger time is subtracted from
the command times used by simulation. The export contains amperes with the
configured physical channel map and polarity and includes the final return to zero.
The model response remains continuous even though one command is issued per turn.
This notebook writes local files; the controller is operated separately.

Image, static and dynamic shims share the waveform directory. Scientific reports
and figures are stored in `OUTPUT_DIR`. The image CSV preserves every spiral-turn state, including leading all-zero
commands. In the default trigger-origin mode there are exactly 65 command rows
and one final zero row for this sequence. Removing zero turns would break the
one-trigger/one-command correspondence during externally triggered playback.
The CSV preview uses the same verified time origin as the model.

Each image export also has a `_zero_time.csv` companion with identical rows,
channel currents and ordering, but a time column filled with zeros. Its playback
requires the intended external-trigger mode; it is not a new optimized waveform.


In [ ]:
trigger_report=sequence_trigger_timing(inputs["paths"]["scan_sequence"],inputs["timing"],channel=TRIGGER_CHANNEL)
csv_edges,csv_timing=waveform_edges_for_origin(model,origin=CSV_TIME_ORIGIN,
    trigger_rf_s=trigger_report["trigger_rf_s"],min_update_interval_s=8e-6)
summary["trigger"]=trigger_report
summary["CSV_timing"]=csv_timing
print("Original time origin:",CSV_TIME_ORIGIN,"| boundaries [ms]:",csv_edges[[0,-1]]*1000)
csv_file=None
csv_zero_time_file=None
if EXPORT_WAVEFORM and solution is not None and solver_ok:
    export=export_waveform(currents,csv_edges,WAVEFORM_PARAMETERS,WAVEFORM_OUTPUT_DIR,
        include_header=True,include_zero_preamble=True,include_final_zero_row=True,
        zero_time_column=False,numbered_filename=True,start_at_first_nonzero=False)
    csv_file=Path(export["csv_file"])
    summary["waveform_export"]=export
    print("Export time-origin shift [ms]:",export["time_origin_shift_s"]*1000,
          "| leading zero states removed:",export["leading_zero_states"])
    zero_time_export=export_zero_time_copy(csv_file)
    csv_zero_time_file=Path(zero_time_export["csv_file"])
    summary["waveform_zero_time_export"]=zero_time_export
    print("CSV with zero time column:",csv_zero_time_file)
    print("CSV :",csv_file)
else:print("CSV export disabled or numerical checks failed.")
(OUTPUT_DIR/"mrzero07_optimization.json").write_text(json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")


## 11. Inspect the final ACDC waveform

Plot the DAC commands and predicted filtered currents for every physical output,
including a zoom on the largest command change, the return to zero and final decay.
`WAVEFORM_PREVIEW_CHANNEL` selects an output explicitly. If a CSV was exported,
the figure reads the actual saved values. Channel map, polarity and time origin
match the export. Filtered current is a model prediction from an initially zero state.


In [ ]:
if csv_file is not None:
    print("Saved CSV :",Path(csv_file).resolve())
else:
    print("No CSV exported for this session.")
    print("Configured output directory:",WAVEFORM_OUTPUT_DIR.resolve())

waveform_report=None
if solution is not None:
    csv_origin_rf_s=trigger_report["trigger_rf_s"] if CSV_TIME_ORIGIN=="first_imaging_trigger" else 0.0
    csv_export=summary.get("waveform_export",{}) if csv_file is not None else {}
    csv_origin_rf_s+=csv_export.get("time_origin_shift_s",0.0)
    fig,waveform_report=plot_acdc_waveform(model,currents,
        origin=CSV_TIME_ORIGIN,trigger_rf_s=trigger_report["trigger_rf_s"],
        channel_map=WAVEFORM_PARAMETERS["channel_map"],
        channel_polarity=WAVEFORM_PARAMETERS["channel_polarity"],
        channel=WAVEFORM_PREVIEW_CHANNEL,csv_file=csv_file,csv_origin_rf_s=csv_origin_rf_s,
        csv_leading_zero_states=csv_export.get("leading_zero_states",0),language="en")
    fig.savefig(OUTPUT_DIR/"mrzero07_acdc_waveform.png",dpi=150)
    show(fig)
    summary["waveform_preview"]=waveform_report
    (OUTPUT_DIR/"mrzero07_optimization.json").write_text(
        json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
else:print("Run optimization to display the final ACDC waveform.")


## 12. Compare measured corrected spiral scans

If `INPUT_DIR/results` contains corrected scans, compare **GRE reference /
spiral without ACDC / spiral with ACDC**. `CORRECTED_SCAN_NAMES=None` selects all
`*scan.dat` files there; an explicit list selects individual acquisitions.
Each scan is checked against the selected spiral sequence and patient slice.
Inaccessible or incompatible scans are reported and skipped.

Every receiver is reconstructed with the same fixed spiral operator before RSS.
One baseline-to-GRE gain, reference normalization, ROI and intensity scale serve
all corrected scans. The GRE is interpolated on the native spiral image grid.
`COMPARISON_REFERENCE_THRESHOLD=0` includes full common coverage. Display-frame
rotation/mirrors change coordinates without moving image pixels.

Measured RMSE/NRMSE are distinct from the simulated optimization objective.
A filename does not prove which waveform was played. Corrected-scan/CSV provenance
must be checked experimentally. Re-running this cell does not rerun optimization.


In [ ]:
import importlib
import spiral_measured_comparison as measured_comparison_tools
compare_measured_scan_set=importlib.reload(measured_comparison_tools).compare_measured_scan_set
measured_comparison=None
results_dir=INPUT_DIR/"results"
if not results_dir.is_dir():
    print("No results directory: measured comparison skipped.")
else:
    corrected_paths=(sorted(results_dir.glob("*scan.dat")) if CORRECTED_SCAN_NAMES is None
                     else [results_dir/name for name in CORRECTED_SCAN_NAMES])
    if not corrected_paths:
        print("No corrected scans selected in",results_dir)
    else:
        measured_comparison=compare_measured_scan_set(inputs,corrected_paths,frame_matrix=O,
            normalize=NORMALIZE_COMPARISON,reference_threshold_fraction=COMPARISON_REFERENCE_THRESHOLD,
            skip_incompatible=True)
        comparison=measured_comparison
        report=comparison["report"]
        if report["skipped_scans"]:
            table([(item["filename"],item["reason"]) for item in report["skipped_scans"]],
                  ("Skipped file","Reason"))
        if not comparison["images"]:
            print("No readable corrected scan matches the selected sequence and reference slice.")
        else:
            units=report["RMSE_units"]
            table([("Shared baseline-to-GRE gain",f"{comparison['gain']:.6g}"),
                   ("GRE normalization scale",f"{comparison['reference_scale']:.6g}"),
                   ("RMSE units",units),
                   ("Common-mask pixels",report["common_pixels"]),
                   ("Pixels excluded for missing coverage",report["excluded_missing_coverage_pixels"])])
            table([(row["label"],f"{row['RMSE']:.6f}",f"{row['NRMSE_percent']:.3f}")
                   for row in comparison["rows"]],("Measured scan vs GRE","RMSE","NRMSE [%]"))
            vmax=max(float(np.nanmax(image)) for image in comparison["images"])
            baseline_row=comparison["rows"][0]
            for index,path in enumerate(map(Path,report["corrected_scan_paths"])):
                corrected_row=comparison["rows"][index+1]
                panels=[comparison["images"][0],comparison["images"][1],comparison["images"][index+2]]
                titles=["GRE reference",f"Scan without ACDC\nRMSE = {baseline_row['RMSE']:.4f}",
                        f"Scan with ACDC\n{path.name}\nRMSE = {corrected_row['RMSE']:.4f}"]
                fig,axes=plt.subplots(1,3,figsize=(18,5),layout="constrained",sharex=True,sharey=True)
                for ax,image,title in zip(axes,panels,titles):
                    ax.set_facecolor("black")
                    im=image_physical(ax,image,comparison["X_m"],comparison["Y_m"],title,vmin=0,vmax=vmax)
                fig.colorbar(im,ax=axes,label=f"Magnitude [{units}]")
                fig.savefig(OUTPUT_DIR/f"mrzero07_measured_comparison_{index+1}.png",dpi=150)
                show(fig)
            np.savez_compressed(OUTPUT_DIR/"measured_comparison.npz",
                images=np.stack(comparison["images"]),labels=np.asarray(comparison["labels"]),
                mask=comparison["mask"],X_m=comparison["X_m"],Y_m=comparison["Y_m"],
                gain=comparison["gain"],reference_scale=comparison["reference_scale"])
        comparison_summary=dict(report,metrics=comparison["rows"])
        (OUTPUT_DIR/"measured_comparison.json").write_text(
            json.dumps(json_safe(comparison_summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
        summary["measured_comparison"]=comparison_summary
        (OUTPUT_DIR/"mrzero07_optimization.json").write_text(
            json.dumps(json_safe(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
